# MiBici - Refactorización del pipeline de aprendizaje automático
## Objetivo
En el notebook 05 desarrollamos el pipeline completo: preparación de la demanda diaria, datos externos, validación temporal y modelos. Funciona, pero el código está repartido en celdas sueltas, algunos valores se repiten en varios lugares y cada ejecución vuelve a descargar o procesar cosas que ya teníamos.

En este notebook reorganizamos ese mismo trabajo por módulos. Cada sección corresponde a un archivo `.py` del paquete `mibici_mlops`. Primero escribimos y probamos aquí las funciones; cuando una sección queda bien, su código se copia al módulo y la celda se reemplaza por el `import`. La lógica es la del 05: no cambiamos el modelo, solo dónde vive cada parte, y al final comprobamos que los resultados sean los mismos.

| Sección | Módulo | Qué contiene |
|---|---|---|
| 2 | `config.py` | Parámetros nuevos de la Fase 2 (se agregan al final del archivo actual) |
| 3 | `dataset.py` | Selección de la ejecución de la Fase 1, demanda diaria e integración |
| 4 | `external.py` (nuevo) | Clima por zona y lugares de interés, con caché |
| 5 | `checks.py` | Contrato de datos y pruebas T1–T9 |
| 6 | `dataset.py` | Etapa de datos completa: reutiliza el Parquet si ya es válido |
| 7 | `features.py` | Variables de historia y calendario |
| 8 | `modeling/evaluate.py` | Walk-forward, métricas y líneas base |
| 9 | `modeling/train.py` | Pipeline de Scikit-Learn, modelos, búsqueda y modelo final |
| 10 | `plots.py` | Gráficas del modelado, guardadas sin abrir ventanas |
| 11 | `tracking.py` | Registro en MLflow y Model Registry |
| 12 | `modeling/predict.py` | Cargar el modelo registrado y predecir el día siguiente |
| 13 | `main.py` | Punto de entrada por etapas |
| 14 | `tests/` | Pruebas de las funciones nuevas |
| 15 | — | Comprobación de equivalencia con el notebook 05 |

**Diseño:** se conserva `MiBiciPipeline` para el procesamiento con estado y se aprovechan los objetos `Pipeline` y `ColumnTransformer` de Scikit-Learn. Las demás transformaciones permanecen como funciones. Cada etapa registra su tiempo para detectar cuellos de botella.

Esta revisión conserva las gráficas exploratorias y el versionado SHA-256 de todos los CSV, además del perfilado de tiempos y el registro en MLflow.

## Requisitos para ejecutar este notebook
- Abrirlo desde `notebooks/` con el paquete instalado en modo editable (`pip install -e .` desde la raíz).
- Librerías: pandas, pyarrow, numpy, matplotlib, scikit-learn 1.3 o posterior, mlflow, requests, osmium, pyproj y shapely.
- Si ya existe `data/processed/mibici_integrated.parquet` y pasa el contrato de datos, la sección 6 lo reutiliza y no se necesita nada más: ni los CSV, ni internet, ni el extracto de OpenStreetMap.
- Si no existe, se necesita una ejecución completa de la Fase 1 en `reports/validation/` y el catálogo de estaciones. El clima y los POI se leen de sus CSV si ya existen; solo se descargan si faltan.
- Con los interruptores de la sección 2 apagados, el modelado tarda alrededor de 30 minutos.

## 1. Preparación del entorno
Importamos las librerías y configuramos dos cosas que se usan en todo el notebook:

- **Backend `Agg` de matplotlib.** Las gráficas se dibujan en memoria, se guardan como PNG y se cierran. Así no se abren ventanas cuando el código corre desde la terminal (`main.py`), y en el notebook mostramos la imagen guardada.
- **`logging` en lugar de `print`** dentro de las funciones. En los módulos los mensajes salen con hora y nivel, y se pueden silenciar o mandar a un archivo sin tocar el código.

De los módulos existentes reutilizamos `config` (rutas y fechas de la Fase 1), `file_sha256` de `dataset.py` y `collect_environment` de `tracking.py`.

In [1]:
import json
import logging
import shutil
import time
from datetime import date, datetime, timezone
from importlib import metadata
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import numpy as np
import osmium
import pandas as pd
import pyarrow.parquet as pq
from pyproj import Transformer
import requests
from shapely.geometry import Point, shape
from shapely.ops import transform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from IPython.display import display
import joblib

from mibici_mlops.tracking import calculate_sha256
from mibici_mlops.tracking import collect_environment

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)
logging.getLogger("mlflow").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logger = logging.getLogger("mibici_mlops")

# Tiempos por etapa (segundos). Se reinician al ejecutar Setup.
stage_times = {}

## 2. Configuración (`config.py`)
En el 05 los parámetros estaban repartidos: las fechas `"2020-01-01"` y `"2026-09-30"` aparecían en tres celdas, la URL de Open-Meteo en dos, los 500 m y la rejilla de 0.05° dentro del código, y los hiperparámetros en varios lugares. Aquí los reunimos en una sola celda. Esta celda se agrega **al final** del `config.py` actual; la primera línea solo importa lo que ya existe en ese archivo y se borra al copiarla.

Qué contiene, por bloques:

- **Rutas.** Conservamos las mismas rutas del 05 (`data/raw/weather`, `data/raw/pois`, `data/processed`) para reutilizar los archivos que ya tenemos.
- **Fuentes externas.** URL, variables y zona horaria de Open-Meteo; tamaño de la rejilla de zonas; URL del extracto de OpenStreetMap, radio, margen y categorías de POI. `DATA_END` se calcula a partir de `STUDY_END`, que en `config.py` es exclusivo (1 de octubre de 2026).
- **Contrato.** Columnas y tipos que debe tener el dataset integrado (sección 8.3 del 05).
- **Problema y validación.** Horizonte, rezagos, ventanas, primer corte, número de ventanas y semilla. `FIRST_CUTOFF` y `TEST_START` reutilizan `TRAIN_END` y `VALIDATION_END` de la Fase 1, que ya son el 1 de enero de 2025 y de 2026.
- **Variables.** Los grupos de columnas y los cuatro conjuntos del 05. El conjunto principal (`calendario_estacion_historia`) tiene 18 variables y no incluye clima.
- **Modelos.** Parámetros iniciales, `BEST_PARAMS`, rejillas e interruptores, igual que en la sección 11.1 del 05.
- **MLflow.** Misma base `mlflow.db` que la Fase 1, un experimento nuevo para el modelado y el nombre del modelo en el Registry.
- **Referencia del 05.** Las cifras contra las que comprobamos la equivalencia en la sección 15.

In [2]:
from mibici_mlops.config import (
    DATA_DIR, FIGURES_DIR, MODELS_DIR, PROCESSED_DATA_DIR, PROJECT_ROOT,
    EXPECTED_MONTHS, STUDY_END, STUDY_START, TRAIN_END, VALIDATION_END,
)

# Fase 2: rutas
VALIDATION_REPORTS_DIR = PROJECT_ROOT / "reports" / "validation"
MODEL_REPORTS_DIR = PROJECT_ROOT / "reports" / "modeling"
MODEL_FIGURES_DIR = FIGURES_DIR / "modeling"
STATIONS_PATH = DATA_DIR / "raw" / "Manual_datos" / "nomenclatura_2026_09.csv"
WEATHER_DIR = DATA_DIR / "raw" / "weather"
WEATHER_PATH = WEATHER_DIR / "weather_daily_zones_2020_2026.csv"
STATION_ZONES_PATH = WEATHER_DIR / "mibici_station_weather_zones.csv"
POI_DIR = DATA_DIR / "raw" / "pois"
OSM_PATH = POI_DIR / "mexico-latest.osm.pbf"
POI_PATH = POI_DIR / "mibici_station_pois.csv"
DAILY_PATH = PROCESSED_DATA_DIR / "mibici_daily.parquet"
INTEGRATED_PATH = PROCESSED_DATA_DIR / "mibici_integrated.parquet"
FINAL_MODEL_PATH = MODELS_DIR / "mibici_final_model.joblib"

# Fase 2: fuentes externas
DATA_START = STUDY_START
DATA_END = STUDY_END - pd.Timedelta(days=1)  # STUDY_END es exclusivo: último día 2026-09-30
WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_COLUMNS = ["temperature_2m_mean", "precipitation_sum"]
WEATHER_TIMEZONE = "America/Mexico_City"
WEATHER_GRID_DEGREES = 0.05
OSM_URL = "https://download.geofabrik.de/north-america/mexico-latest.osm.pbf"
POI_RADIUS_METERS = 500
POI_BBOX_MARGIN = 0.01
METRIC_CRS = "EPSG:32613"  # UTM 13N, distancias en metros
POI_CATEGORIES = {
    "education": {"amenity": ["school", "university", "college"]},
    "transport": {"railway": ["station", "tram_stop"], "public_transport": ["station"]},
    "commerce": {"shop": ["supermarket", "mall"], "amenity": ["marketplace"]},
    "work": {"office": None},
    "health": {"amenity": ["hospital", "clinic"]},
    "recreation": {"leisure": ["park", "sports_centre", "pitch"]},
}
POI_COLUMNS = [f"{category}_count" for category in POI_CATEGORIES]

# Fase 2: contrato del dataset integrado
CONTRACT_DTYPES = {
    "Origen_Id": "int64",
    "Fecha": "datetime64[us]",
    "target": "int64",
    "day_of_week": "int32",
    "month": "int32",
    "weather_zone": "int64",
    "temperature_2m_mean": "float64",
    "precipitation_sum": "float64",
    **{column: "int64" for column in POI_COLUMNS},
}

# Fase 2: problema y validación temporal
TARGET_COLUMN = "target"
HORIZON_DAYS = 1  # predecimos el día siguiente
LAGS = [1, 7, 14]  # deben ser >= HORIZON_DAYS
ROLLING_WINDOWS = [7, 28]
FIRST_CUTOFF = TRAIN_END  # primer mes evaluado: enero de 2025
TEST_START = VALIDATION_END  # 2026 se reserva para la prueba final
N_FOLDS = 12
TUNING_FOLD_STEP = 2  # la búsqueda usa una de cada dos ventanas
SEED = 42

# Fase 2: variables
STATION_ID_COLUMNS = ["Origen_Id"]
ONEHOT_COLUMNS = ["day_of_week", "month", "weather_zone"]
CALENDAR_COLUMNS = ["day_of_week", "month", "day_of_month"]
STATION_COLUMNS = ["weather_zone"] + POI_COLUMNS
HISTORY_COLUMNS = [f"lag_{lag}" for lag in LAGS] + [
    f"{prefix}_{window}" for window in ROLLING_WINDOWS for prefix in ("roll_mean", "active_days")
]
FEATURE_SETS = {
    "original_11_variables": ["day_of_week", "month"] + STATION_COLUMNS + WEATHER_COLUMNS,
    "calendario_estacion": STATION_ID_COLUMNS + CALENDAR_COLUMNS + STATION_COLUMNS,
    "calendario_estacion_historia": (
        STATION_ID_COLUMNS + CALENDAR_COLUMNS + STATION_COLUMNS + HISTORY_COLUMNS
    ),
    "historia_y_clima_observado": (
        STATION_ID_COLUMNS + CALENDAR_COLUMNS + STATION_COLUMNS + HISTORY_COLUMNS + WEATHER_COLUMNS
    ),
}
MAIN_FEATURE_SET = "calendario_estacion_historia"
FEATURE_COLUMNS = FEATURE_SETS[MAIN_FEATURE_SET]

# Fase 2: modelos
MODEL_NAMES = ["Random Forest", "HistGradientBoosting"]
RF_INITIAL_PARAMS = {
    "n_estimators": 100,
    "max_depth": 15,
    "min_samples_leaf": 5,
    "max_samples": 0.5,  # cada árbol usa 50 % de las filas
    "random_state": SEED,
    "n_jobs": -1,
}
HGB_INITIAL_PARAMS = {
    "max_iter": 100,
    "max_leaf_nodes": 31,
    "learning_rate": 0.1,
    "random_state": SEED,
}
# Se usan cuando RUN_GRID_SEARCH = False. Random Forest: mejor configuración de la búsqueda
# de la versión anterior. HistGradientBoosting: punto de partida, todavía sin búsqueda propia.
FINAL_MODEL = "Random Forest"
BEST_PARAMS = {
    "Random Forest": {
        "n_estimators": 150,
        "max_depth": 15,
        "min_samples_leaf": 5,
        "max_features": 0.7,
    },
    "HistGradientBoosting": {
        "learning_rate": 0.1,
        "max_iter": 200,
        "max_leaf_nodes": 31,
        "loss": "poisson",
    },
}
PARAMETER_GRIDS = {
    "Random Forest": [
        {
            "model__n_estimators": [150],
            "model__max_depth": [12, 15, 18],
            "model__min_samples_leaf": [5],
            "model__max_features": [0.7, 1.0],
        },
        {
            "model__n_estimators": [150],
            "model__max_depth": [15],
            "model__min_samples_leaf": [3, 8],
            "model__max_features": [0.7],
        },
    ],
    "HistGradientBoosting": {
        "model__learning_rate": [0.05, 0.1],
        "model__max_iter": [200, 400],
        "model__max_leaf_nodes": [31, 63],
        "model__loss": ["squared_error", "poisson"],
    },
}

# Fase 2: interruptores de las partes que tardan
FORCE_REBUILD_DATA = False  # True: reconstruir el Parquet integrado aunque sea válido
ALLOW_UNVERIFIED_CACHE = False  # False: no reutilizar caché sin manifiesto verificable
COMPARE_FEATURE_SETS = False  # comparar los cuatro conjuntos de variables (~15 min)
RUN_WEATHER_REFERENCE = False  # modelo con clima observado (~4 min)
RUN_GRID_SEARCH = False  # búsqueda de hiperparámetros (~1 h)

# Fase 2: MLflow
MLFLOW_TRACKING_URI = f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
MLFLOW_ARTIFACT_DIR = PROJECT_ROOT / "mlruns"
MODELING_EXPERIMENT = "MiBici_Fase2_Modelado"
REGISTERED_MODEL_NAME = "mibici_salidas_diarias"
MODEL_ALIAS = "champion"

# Fase 2: resultados del notebook 05 (MAE) para comprobar la equivalencia
REFERENCE_RESULTS = {
    "Ingenuo estacional (lag_7)": 7.224,
    "Promedio estación-día": 7.703,
    "Promedio global por día (versión anterior)": 22.986,
    "HistGradientBoosting | calendario_estacion_historia": 5.440,
    "Random Forest | calendario_estacion_historia": 5.456,
    "Modelo final, 12 ventanas de 2025": 5.438,
    "Modelo final, prueba 2026 (todas)": 5.896,
    "Ingenuo estacional, prueba 2026 (todas)": 7.957,
    "Modelo final, prueba 2026 (nuevas)": 3.511,
}

In [3]:
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Periodo de datos: {DATA_START.date()} a {DATA_END.date()}")
print(f"Primer corte: {FIRST_CUTOFF.date()} | Inicio de la prueba: {TEST_START.date()}")
print(f"Conjunto principal: {MAIN_FEATURE_SET} ({len(FEATURE_COLUMNS)} variables)")
print(f"¿Hay clima en el conjunto principal? {any(c in FEATURE_COLUMNS for c in WEATHER_COLUMNS)}")

Raíz del proyecto: /home/josel/mlops/equipo-8-mibici-mlops
Periodo de datos: 2020-01-01 a 2026-09-30
Primer corte: 2025-01-01 | Inicio de la prueba: 2026-01-01
Conjunto principal: calendario_estacion_historia (18 variables)
¿Hay clima en el conjunto principal? False


## 3. Preparación de la demanda diaria (`dataset.py`)
El módulo `dataset.py` de la Fase 1 contiene funciones de preparación (relleno de horas con cero y separación horaria). No las tocamos: siguen sirviendo para `main.py` de la Fase 1. Agregamos las funciones de la Fase 2, que salen de las secciones 3 y 4 del 05.

### 3.1 Selección y validación de la ejecución de la Fase 1
En el 05 esto ocupaba seis celdas (3.5 a 3.8). Lo reunimos en dos funciones:

- `find_latest_valid_run`: busca en `reports/validation/` las carpetas con `hourly_demand.parquet` y `manifest.json`, se queda con las que procesaron los 81 archivos del periodo completo y devuelve la más reciente. Si no hay ninguna, devuelve `None`.
- `check_processed_run`: repite las cuatro validaciones del 05 y devuelve una tabla con el resultado de cada una: la demanda horaria suma lo mismo que los viajes conservados, están los 81 Parquet mensuales, todos se pueden abrir y sus filas suman los viajes conservados.

Las celdas del 05 que leían el código fuente con `ast` y las de limpieza de ejecuciones anteriores (3.9 a 3.11) no pasan aquí: eran revisión, no parte del pipeline.

In [4]:
def find_latest_valid_run(validation_dir: Path, expected_files: int) -> Path | None:
    # Devolver la ejecución completa más reciente de la Fase 1.
    candidates = []
    for hourly_path in validation_dir.rglob("hourly_demand.parquet"):
        manifest_path = hourly_path.parent / "manifest.json"
        if not manifest_path.is_file() or hourly_path.stat().st_size == 0:
            continue
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if (
            manifest.get("files_processed") == expected_files
            and manifest.get("full_period") is True
            and manifest.get("finished_utc")
        ):
            candidates.append((manifest["finished_utc"], hourly_path.parent))
    if not candidates:
        return None
    return max(candidates)[1]


def check_processed_run(run_dir: Path, expected_months: pd.PeriodIndex) -> pd.Series:
    # Validar que la ejecución sea consistente antes de reutilizarla.
    manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
    kept_rows = manifest["rows"]["kept_rows"]
    hourly = pq.read_table(run_dir / "hourly_demand.parquet", columns=["Cantidad_viajes"])
    hourly_total = int(hourly.column("Cantidad_viajes").to_numpy().sum())

    expected = {f"datos_abiertos_{m.year}_{m.month:02d}.parquet" for m in expected_months}
    trip_files = sorted((run_dir / "trips").glob("*.parquet"))
    found = {path.name for path in trip_files if path.stat().st_size > 0}

    parquet_rows, unreadable = 0, []
    for path in trip_files:
        try:
            parquet_rows += pq.ParquetFile(path).metadata.num_rows
        except (OSError, ValueError):
            unreadable.append(path.name)

    return pd.Series({
        "Demanda horaria = viajes conservados": hourly_total == kept_rows,
        "Cobertura mensual completa": found == expected,
        "Parquet legibles": not unreadable,
        "Filas de Parquet = viajes conservados": parquet_rows == kept_rows,
    })

def cache_manifest_path(path: Path) -> Path:
    return path.with_suffix(".manifest.json")


def build_cache_signature(run_dir: Path) -> dict:
    # Identificar las entradas y reglas que determinan el dataset integrado.
    inputs = {
        "hourly_demand": run_dir / "hourly_demand.parquet",
        "station_catalog": STATIONS_PATH,
        "weather": WEATHER_PATH,
        "station_zones": STATION_ZONES_PATH,
        "poi": POI_PATH,
    }
    return {
        "schema_version": 1,
        "study_start": str(DATA_START.date()),
        "study_end": str(DATA_END.date()),
        "weather_grid_degrees": WEATHER_GRID_DEGREES,
        "poi_radius_meters": POI_RADIUS_METERS,
        "poi_categories": POI_CATEGORIES,
        "contract_dtypes": CONTRACT_DTYPES,
        "sources": {
            name: {"sha256": calculate_sha256(path), "size": path.stat().st_size}
            for name, path in inputs.items()
        },
    }


def cache_is_current(path: Path, signature: dict) -> bool:
    manifest_path = cache_manifest_path(path)
    if not path.is_file() or not manifest_path.is_file():
        return False
    try:
        saved = json.loads(manifest_path.read_text(encoding="utf-8"))
        return saved.get("signature") == signature and saved.get("sha256") == calculate_sha256(path)
    except (OSError, ValueError, json.JSONDecodeError):
        return False


def save_cache_manifest(path: Path, signature: dict) -> None:
    manifest = {"signature": signature, "sha256": calculate_sha256(path)}
    cache_manifest_path(path).write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )

### 3.2 Demanda diaria, coordenadas e integración de fuentes
Estas tres funciones son las del 05 casi sin cambios:

- `build_daily_demand` (antes `prepare_daily_demand`): agrupa los viajes por estación y fecha, renombra la suma como `target` y agrega `day_of_week` y `month`. Los días sin viajes **no** se rellenan con cero: la estación simplemente no tiene fila ese día. Cambiamos `dt.dayofweek` por `dt.day_of_week`, que es el nombre que recomienda pandas; el resultado es el mismo.
- `load_station_coordinates`: lee el catálogo de estaciones y conserva solo las que tienen latitud y longitud.
- `integrate_daily_sources`: une la zona meteorológica por estación, el clima por zona y fecha, y los POI por estación. Cada unión usa `validate="many_to_one"`, así que si una tabla trajera claves repetidas, pandas lanza un error en lugar de duplicar filas.

In [5]:
def build_daily_demand(hourly_data: pd.DataFrame) -> pd.DataFrame:
    # Pasar de viajes por estación y hora a viajes por estación y día.
    daily = hourly_data.copy()
    daily["Fecha"] = daily["Hora"].dt.normalize()
    daily = daily.groupby(["Origen_Id", "Fecha"], as_index=False)["Cantidad_viajes"].sum()
    daily = daily.rename(columns={"Cantidad_viajes": "target"})
    daily["day_of_week"] = daily["Fecha"].dt.day_of_week
    daily["month"] = daily["Fecha"].dt.month
    return daily


def load_station_coordinates(stations_path: Path) -> pd.DataFrame:
    # Leer el catálogo y conservar las estaciones con coordenadas.
    stations = pd.read_csv(stations_path)
    stations = stations[["id", "latitude", "longitude"]].rename(columns={"id": "Origen_Id"})
    return stations.dropna(subset=["latitude", "longitude"])


def integrate_daily_sources(daily_data, weather_data, weather_zones, poi_data):
    # Unir clima por zona y fecha, y POI por estación, sin duplicar filas.
    integrated = daily_data.copy()
    integrated["Fecha"] = pd.to_datetime(integrated["Fecha"])
    integrated = integrated.merge(
        weather_zones, on="Origen_Id", how="left", validate="many_to_one"
    )
    integrated = integrated.merge(
        weather_data[["time", "weather_zone"] + WEATHER_COLUMNS],
        left_on=["Fecha", "weather_zone"],
        right_on=["time", "weather_zone"],
        how="left",
        validate="many_to_one",
    )
    integrated = integrated.drop(columns=["time"])
    return integrated.merge(poi_data, on="Origen_Id", how="left", validate="many_to_one")

## 4. Fuentes externas (`external.py`)
Este módulo es nuevo. En el 05 el clima se descargaba en **cada** ejecución y sobrescribía el CSV, aunque los datos históricos no cambian. Los POI sí revisaban si el extracto de OpenStreetMap ya existía, pero el cálculo de conteos se repetía siempre. Aquí las dos fuentes siguen la misma regla: si el CSV ya existe, se lee; solo si falta (o si se pide con `force=True`) se descarga y se calcula.

### 4.1 Clima por zona (Open-Meteo)
- `assign_weather_zones`: redondea las coordenadas a una rejilla de 0.05° (unos 5 km) y numera las celdas con estaciones. Cada celda es una zona meteorológica.
- `weather_zone_points`: calcula el centro de cada zona, que es el punto que se consulta.
- `download_zone_weather`: descarga la temperatura media y la precipitación diaria de cada zona y valida que estén todas las fechas, sin faltantes ni duplicados. Son las mismas validaciones de la 5.4 del 05.
- `load_or_download_weather`: si existen los dos CSV (clima y relación estación-zona), los lee; si no, los genera y los guarda.

In [6]:
def assign_weather_zones(stations: pd.DataFrame, grid_degrees: float) -> pd.DataFrame:
    # Asignar cada estación a una celda de la rejilla.
    zones = stations.drop_duplicates(subset="Origen_Id").copy()
    zones["latitude_zone"] = (zones["latitude"] / grid_degrees).round() * grid_degrees
    zones["longitude_zone"] = (zones["longitude"] / grid_degrees).round() * grid_degrees
    zones["weather_zone"] = zones.groupby(["latitude_zone", "longitude_zone"]).ngroup() + 1
    return zones


def weather_zone_points(zones: pd.DataFrame) -> pd.DataFrame:
    # Coordenadas centrales de cada zona.
    return zones.groupby("weather_zone", as_index=False).agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean"),
        station_count=("Origen_Id", "count"),
    )


def download_zone_weather(points, start, end, url, variables, timezone_name):
    # Descargar el clima diario de cada zona y validar el resultado.
    records = []
    for zone in points.itertuples(index=False):
        params = {
            "latitude": float(zone.latitude),
            "longitude": float(zone.longitude),
            "start_date": start.strftime("%Y-%m-%d"),
            "end_date": end.strftime("%Y-%m-%d"),
            "daily": ",".join(variables),
            "timezone": timezone_name,
        }
        response = requests.get(url, params=params, timeout=120)
        response.raise_for_status()
        zone_weather = pd.DataFrame(response.json()["daily"])
        zone_weather["weather_zone"] = zone.weather_zone
        zone_weather["latitude"] = zone.latitude
        zone_weather["longitude"] = zone.longitude
        records.append(zone_weather)
        logger.info("Zona %s: %s días descargados", zone.weather_zone, len(zone_weather))

    weather = pd.concat(records, ignore_index=True)
    weather = weather[["time", "weather_zone", "latitude", "longitude"] + variables]
    expected_rows = len(pd.date_range(start, end)) * len(points)
    if len(weather) != expected_rows:
        raise ValueError("La cantidad de registros meteorológicos no coincide.")
    if weather.isna().any().any():
        raise ValueError("Se encontraron valores meteorológicos faltantes.")
    if weather.duplicated(["time", "weather_zone"]).any():
        raise ValueError("Se encontraron fechas duplicadas por zona.")
    return weather


def load_or_download_weather(stations: pd.DataFrame, force: bool = False):
    # Leer el clima guardado o descargarlo si todavía no existe.
    if WEATHER_PATH.is_file() and STATION_ZONES_PATH.is_file() and not force:
        weather = pd.read_csv(WEATHER_PATH, parse_dates=["time"])
        zones = pd.read_csv(STATION_ZONES_PATH)
        expected_ids = set(stations["Origen_Id"])
        if (
            expected_ids.issubset(set(zones["Origen_Id"]))
            and not zones["Origen_Id"].duplicated().any()
            and not weather.duplicated(["time", "weather_zone"]).any()
            and set(WEATHER_COLUMNS).issubset(weather.columns)
            and weather["time"].min() <= DATA_START
            and weather["time"].max() >= DATA_END
        ):
            logger.info("Clima reutilizado: %s", WEATHER_PATH.name)
            return weather, zones
        logger.warning("El caché meteorológico está incompleto; se reconstruye.")

    zones = assign_weather_zones(stations, WEATHER_GRID_DEGREES)
    weather = download_zone_weather(
        weather_zone_points(zones), DATA_START, DATA_END,
        WEATHER_URL, WEATHER_COLUMNS, WEATHER_TIMEZONE,
    )
    WEATHER_DIR.mkdir(parents=True, exist_ok=True)
    weather.to_csv(WEATHER_PATH, index=False)
    zones[["Origen_Id", "weather_zone"]].to_csv(STATION_ZONES_PATH, index=False)
    weather["time"] = pd.to_datetime(weather["time"])
    return weather, zones[["Origen_Id", "weather_zone"]]

### 4.2 Lugares de interés (OpenStreetMap)
Aquí sí usamos clases, porque Osmium lo pide: para recorrer el archivo `.pbf` hay que heredar de `osmium.SimpleHandler` y definir qué hacer con cada nodo o área. En el 05 eran dos clases con la misma lógica de clasificación copiada; la sacamos a `matching_categories` para no repetirla.

- `matching_categories`: revisa las etiquetas de un elemento (por ejemplo `amenity=school`) y devuelve las categorías a las que pertenece.
- `POINodeExtractor` y `POIAreaExtractor`: guardan los nodos y las áreas (parques, hospitales, plazas) dentro del área de MiBici. Para las áreas se conserva el polígono completo.
- `download_osm_extract`: descarga el extracto de Geofabrik solo si no existe.
- `count_pois_near_stations`: proyecta todo a UTM 13N para medir en metros y cuenta, por estación y categoría, los lugares a 500 m o menos. Para las áreas mide la distancia al borde del polígono, no a su centro.
- `load_or_build_poi_counts`: si el CSV de conteos existe, lo lee; si no, hace todo lo anterior y lo guarda.

In [7]:
def matching_categories(tags: dict, categories: dict) -> list[str]:
    # Categorías cuyas etiquetas coinciden con las del elemento.
    matched = []
    for category, filters in categories.items():
        for key, values in filters.items():
            if key in tags and (values is None or tags[key] in values):
                matched.append(category)
                break
    return matched


def inside_bbox(longitude: float, latitude: float, bbox: list[float]) -> bool:
    return bbox[0] <= longitude <= bbox[2] and bbox[1] <= latitude <= bbox[3]


class POINodeExtractor(osmium.SimpleHandler):
    # Lugares de interés representados como puntos.
    def __init__(self, bbox, categories):
        super().__init__()
        self.bbox = bbox
        self.categories = categories
        self.records = []

    def node(self, node):
        if not node.location.valid():
            return
        longitude, latitude = node.location.lon, node.location.lat
        if not inside_bbox(longitude, latitude, self.bbox):
            return
        tags = dict(node.tags)
        for category in matching_categories(tags, self.categories):
            self.records.append({
                "osm_type": "node", "osm_id": node.id, "category": category,
                "name": tags.get("name"), "latitude": latitude, "longitude": longitude,
                "geometry": Point(longitude, latitude),
            })


class POIAreaExtractor(osmium.SimpleHandler):
    # Lugares de interés representados como polígonos.
    def __init__(self, bbox, categories):
        super().__init__()
        self.bbox = bbox
        self.categories = categories
        self.records = []
        self.factory = osmium.geom.GeoJSONFactory()

    def area(self, area):
        tags = dict(area.tags)
        matched = matching_categories(tags, self.categories)
        if not matched:
            return
        try:
            polygon = shape(json.loads(self.factory.create_multipolygon(area)))
        except (RuntimeError, ValueError):
            return
        if polygon.is_empty:
            return
        center = polygon.representative_point()
        if not inside_bbox(center.x, center.y, self.bbox):
            return
        for category in matched:
            self.records.append({
                "osm_type": "area", "osm_id": area.orig_id(), "category": category,
                "name": tags.get("name"), "latitude": center.y, "longitude": center.x,
                "geometry": polygon,
            })


def download_osm_extract(url: str, path: Path) -> Path:
    # Descargar el extracto de OpenStreetMap solo si no existe.
    if path.is_file() and path.stat().st_size > 0:
        logger.info("Extracto OSM reutilizado: %s", path.name)
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                file.write(chunk)
    logger.info("Extracto OSM descargado: %s", path)
    return path


def extract_pois(osm_path: Path, bbox: list[float], categories: dict) -> pd.DataFrame:
    # Leer nodos y áreas del extracto y quitar duplicados.
    nodes = POINodeExtractor(bbox, categories)
    nodes.apply_file(str(osm_path))
    areas = POIAreaExtractor(bbox, categories)
    areas.apply_file(str(osm_path), locations=True, idx="flex_mem")
    pois = pd.DataFrame(nodes.records + areas.records)
    return pois.drop_duplicates(subset=["osm_type", "osm_id", "category"])


def count_pois_near_stations(stations, pois, categories, radius, metric_crs) -> pd.DataFrame:
    # Contar por estación y categoría los POI dentro del radio.
    transformer = Transformer.from_crs("EPSG:4326", metric_crs, always_xy=True)
    projected = [transform(transformer.transform, geometry) for geometry in pois["geometry"]]
    poi_categories = pois["category"].to_numpy()
    rows = []
    for station in stations.itertuples(index=False):
        station_point = Point(*transformer.transform(station.longitude, station.latitude))
        nearby = [
            category for category, geometry in zip(poi_categories, projected)
            if station_point.distance(geometry) <= radius
        ]
        counts = pd.Series(nearby, dtype="object").value_counts()
        rows.append({
            "Origen_Id": station.Origen_Id,
            **{f"{category}_count": int(counts.get(category, 0)) for category in categories},
        })
    return pd.DataFrame(rows)


def load_or_build_poi_counts(stations: pd.DataFrame, force: bool = False) -> pd.DataFrame:
    # Leer los conteos guardados o calcularlos desde OpenStreetMap.
    if POI_PATH.is_file() and not force:
        logger.info("POI reutilizados: %s", POI_PATH.name)
        return pd.read_csv(POI_PATH)

    stations = stations.drop_duplicates(subset="Origen_Id")
    bbox = [
        stations["longitude"].min() - POI_BBOX_MARGIN,
        stations["latitude"].min() - POI_BBOX_MARGIN,
        stations["longitude"].max() + POI_BBOX_MARGIN,
        stations["latitude"].max() + POI_BBOX_MARGIN,
    ]
    download_osm_extract(OSM_URL, OSM_PATH)
    pois = extract_pois(OSM_PATH, bbox, POI_CATEGORIES)
    counts = count_pois_near_stations(
        stations, pois, POI_CATEGORIES, POI_RADIUS_METERS, METRIC_CRS
    )
    POI_DIR.mkdir(parents=True, exist_ok=True)
    counts.to_csv(POI_PATH, index=False)
    return counts

## 5. Contrato y pruebas de calidad (`checks.py`)
`checks.py` hoy solo tiene `check_counts`, de la Fase 1. Le agregamos tres funciones con las reglas de las secciones 5.5.9, 7.4, 8.3, 8.4 y 8.5 del 05. En el 05 esas reglas estaban escritas dos veces (el contrato con `assert` y luego otra vez en la tabla T1–T9); aquí quedan una sola vez:

- `validate_contract`: aplica las pruebas T1 a T9 y devuelve una tabla con el resultado de cada una. No detiene la ejecución; quien la llama decide qué hacer si algo falla. En T2, para `Fecha` aceptamos cualquier resolución (`ns` o `us`), porque depende de la versión de pandas con la que se guardó el Parquet.
- `check_poi_counts`: revisa que haya conteos para todas las estaciones, sin duplicados, faltantes, negativos ni decimales.
- `check_integration`: comprueba que integrar el clima y los POI no cambió la demanda: mismas filas, mismas estaciones y fechas, mismo total de viajes.

In [8]:
def validate_contract(data: pd.DataFrame, dtypes: dict, poi_columns: list[str]) -> pd.DataFrame:
    # Pruebas T1-T9 del dataset integrado.
    has_schema = set(data.columns) == set(dtypes)
    if not has_schema:
        missing = sorted(set(dtypes) - set(data.columns))
        extra = sorted(set(data.columns) - set(dtypes))
        raise ValueError(f"Esquema inesperado. Faltan: {missing}; sobran: {extra}")
    text_columns = data.select_dtypes(include="object").columns
    empty_texts = sum(data[c].astype(str).str.strip().eq("").sum() for c in text_columns)
    checks = {
        "T1 - Esquema": has_schema,
        "T2 - Tipos": has_schema and all(
            pd.api.types.is_datetime64_any_dtype(data[c]) if t.startswith("datetime64")
            else str(data[c].dtype) == t
            for c, t in dtypes.items()
        ),
        "T3 - Textos vacíos": empty_texts == 0,
        "T4 - Valores inválidos": has_schema and (
            (data["Origen_Id"] > 0).all()
            and (data["weather_zone"] >= 0).all()
            and (data[poi_columns] >= 0).all().all()
        ),
        "T5 - Nulos": not data.isna().any().any(),
        "T6 - Rangos": has_schema and (
            data["day_of_week"].between(0, 6).all()
            and data["month"].between(1, 12).all()
            and (data["precipitation_sum"] >= 0).all()
        ),
        "T7 - Fechas consistentes": has_schema and (
            data["Fecha"].dt.day_of_week.eq(data["day_of_week"]).all()
            and data["Fecha"].dt.month.eq(data["month"]).all()
        ),
        "T8 - Duplicados": not data.duplicated(["Origen_Id", "Fecha"]).any(),
        "T9 - Objetivo": (data["target"] >= 0).all()
        and pd.api.types.is_integer_dtype(data["target"]),
    }
    return pd.DataFrame({"test": list(checks), "passed": [bool(v) for v in checks.values()]})


def check_poi_counts(poi_data: pd.DataFrame, station_ids: set) -> None:
    # Validar los conteos de POI antes de integrarlos.
    count_columns = poi_data.columns.drop("Origen_Id")
    if set(poi_data["Origen_Id"]) != set(station_ids):
        raise ValueError("Los identificadores de los POI no coinciden con las estaciones.")
    if poi_data["Origen_Id"].duplicated().any():
        raise ValueError("Hay estaciones duplicadas en los POI.")
    if poi_data.isna().any().any():
        raise ValueError("Hay conteos de POI faltantes.")
    if (poi_data[count_columns] < 0).any().any():
        raise ValueError("Hay conteos de POI negativos.")
    if not all(pd.api.types.is_integer_dtype(poi_data[c]) for c in count_columns):
        raise ValueError("Hay conteos de POI que no son enteros.")


def check_integration(daily_data: pd.DataFrame, integrated: pd.DataFrame) -> None:
    # La integración no debe cambiar filas, claves ni viajes.
    keys = ["Origen_Id", "Fecha", "target"]
    if len(integrated) != len(daily_data):
        raise ValueError("La integración cambió la cantidad de registros.")
    if integrated["target"].sum() != daily_data["target"].sum():
        raise ValueError("La integración cambió el total de viajes.")
    if not integrated[keys].equals(daily_data[keys]):
        raise ValueError("La integración cambió estaciones, fechas o viajes.")

### 3.3 Versionado SHA-256 de archivos CSV

Guardamos un inventario de los CSV encontrados en `data/`, con su SHA-256, tamaño y número de versión. También se versionan los CSV generados en `reports/modeling/`. Los originales no se modifican. El inventario incluye archivos disponibles, aunque algunos no hayan participado en el entrenamiento; el manifiesto de procedencia los distingue.


In [9]:
CSV_VERSION_DIR = PROJECT_ROOT / "reports" / "data_versioning"
CSV_VERSION_INDEX = CSV_VERSION_DIR / "csv_versions.json"
CSV_VERSION_TABLE = CSV_VERSION_DIR / "csv_versions.csv"
CSV_SNAPSHOT_MAX_BYTES = 10 * 1024 * 1024


def version_csv_sources(root: Path, version_dir: Path) -> pd.DataFrame:
    """Inventariar todos los CSV sin alterar los archivos originales."""
    version_dir.mkdir(parents=True, exist_ok=True)
    index_path = version_dir / "csv_versions.json"
    previous = json.loads(index_path.read_text(encoding="utf-8")) if index_path.is_file() else {}
    current = {}
    rows = []
    for path in sorted(p for p in root.rglob("*") if p.suffix.lower() == ".csv"):
        if not path.is_file():
            continue
        relative = path.relative_to(PROJECT_ROOT).as_posix()
        stat = path.stat()
        old = previous.get(relative, {})
        if old.get("size_bytes") == stat.st_size and old.get("mtime_ns") == stat.st_mtime_ns:
            sha256 = old["sha256"]
        else:
            sha256 = calculate_sha256(path)
        history = old.get("versions", [])
        if sha256 not in history:
            history.append(sha256)
        current[relative] = {
            "status": "present",
            "sha256": sha256,
            "size_bytes": stat.st_size,
            "mtime_ns": stat.st_mtime_ns,
            "versions": history,
        }
        rows.append({
            "file": relative, "sha256": sha256, "size_bytes": stat.st_size,
            "version": history.index(sha256) + 1, "known_versions": len(history),
        })
        # Los archivos originales grandes no se copian ni se agregan a Git.
        if stat.st_size <= CSV_SNAPSHOT_MAX_BYTES and "data/raw/datos_abiertos_" not in relative:
            snapshot = version_dir / "snapshots" / path.stem / f"{path.stem}_{sha256[:12]}.csv"
            if not snapshot.is_file():
                snapshot.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, snapshot)
    # Conservar el historial de archivos que ya no están presentes.
    for relative, old in previous.items():
        if relative not in current:
            current[relative] = {**old, "status": "missing"}
    index_path.write_text(json.dumps(current, indent=2, ensure_ascii=False), encoding="utf-8")
    inventory = pd.DataFrame(rows, columns=["file", "sha256", "size_bytes", "version", "known_versions"])
    inventory.to_csv(version_dir / "csv_versions.csv", index=False)
    return inventory

## 6. Etapa de datos completa (`dataset.py`)
Esta función une las secciones 3 a 5 y es la que llamará `main.py`. Decide si hay que reconstruir los datos:

1. Si ya existe `mibici_integrated.parquet` y pasa las nueve pruebas del contrato, lo lee y termina. Es el caso normal: no se leen CSV, no se descarga nada y tarda unos segundos.
2. Si no existe, si falla el contrato o si `FORCE_REBUILD_DATA = True`, busca la última ejecución válida de la Fase 1, construye la demanda diaria, obtiene el clima y los POI (de sus CSV o descargándolos), integra, valida y guarda los dos Parquet.

Si no encuentra una ejecución válida de la Fase 1, lanza un error que dice qué hacer, en lugar de reprocesar los 81 CSV sin avisar.

**Caché verificable:** la primera ejecución después de este cambio puede reconstruir el integrado para crear su manifiesto. En las siguientes ejecuciones, se reutiliza si coinciden las entradas, las reglas y la huella SHA-256. No se considera suficiente que el archivo exista.

In [10]:
def prepare_model_data(force: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Reutilizar el integrado solo si su contenido y sus entradas están verificados.
    run_dir = find_latest_valid_run(VALIDATION_REPORTS_DIR, len(EXPECTED_MONTHS))
    if run_dir is None:
        raise FileNotFoundError(
            "No hay una ejecución completa de la Fase 1 en reports/validation. "
            "Ejecutar primero el procesamiento de la Fase 1 (MiBiciPipeline)."
        )
    run_checks = check_processed_run(run_dir, EXPECTED_MONTHS)
    if not run_checks.all():
        raise ValueError(f"La ejecución {run_dir.name} no pasa las validaciones:\n{run_checks}")
    logger.info("Ejecución de la Fase 1 utilizada: %s", run_dir.name)

    sources_ready = all(path.is_file() for path in (
        WEATHER_PATH, STATION_ZONES_PATH, POI_PATH, STATIONS_PATH,
    ))
    signature = build_cache_signature(run_dir) if sources_ready else None
    if INTEGRATED_PATH.is_file() and not force:
        if signature is not None and cache_is_current(INTEGRATED_PATH, signature):
            integrated = pd.read_parquet(INTEGRATED_PATH)
            report = validate_contract(integrated, CONTRACT_DTYPES, POI_COLUMNS)
            if report["passed"].all():
                logger.info("Dataset integrado reutilizado: %s", INTEGRATED_PATH)
                return integrated, report
            logger.warning("El integrado no pasa las pruebas de calidad; se reconstruye.")
        elif ALLOW_UNVERIFIED_CACHE:
            integrated = pd.read_parquet(INTEGRATED_PATH)
            report = validate_contract(integrated, CONTRACT_DTYPES, POI_COLUMNS)
            if report["passed"].all():
                logger.warning("Reutilizando Parquet sin procedencia verificada: %s", INTEGRATED_PATH)
                return integrated, report
        else:
            logger.info("No hay manifiesto vigente; se reconstruye el integrado una vez.")

    daily = build_daily_demand(pd.read_parquet(run_dir / "hourly_demand.parquet"))
    PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
    daily.to_parquet(DAILY_PATH, index=False)

    stations = load_station_coordinates(STATIONS_PATH)
    stations = stations[stations["Origen_Id"].isin(daily["Origen_Id"])]
    weather, weather_zones = load_or_download_weather(stations)
    poi_data = load_or_build_poi_counts(stations)
    check_poi_counts(poi_data, set(daily["Origen_Id"]))

    integrated = integrate_daily_sources(daily, weather, weather_zones, poi_data)
    check_integration(daily, integrated)
    report = validate_contract(integrated, CONTRACT_DTYPES, POI_COLUMNS)
    if not report["passed"].all():
        raise ValueError(f"El dataset integrado no pasa el contrato:\n{report}")
    integrated.to_parquet(INTEGRATED_PATH, index=False)
    save_cache_manifest(INTEGRATED_PATH, build_cache_signature(run_dir))
    logger.info("Dataset integrado guardado: %s", INTEGRATED_PATH)
    return integrated, report

### 6.1 Ejecución de la etapa de datos
Ejecutamos la etapa y mostramos el resultado de las pruebas. Medimos el tiempo para registrarlo después en MLflow.

In [11]:
start = time.perf_counter()
integrated_data, quality_report = prepare_model_data(force=FORCE_REBUILD_DATA)
data_seconds = time.perf_counter() - start

display(quality_report)
print(f"Dimensiones: {integrated_data.shape}")
print(f"Periodo: {integrated_data['Fecha'].min().date()} a {integrated_data['Fecha'].max().date()}")
print(f"Estaciones: {integrated_data['Origen_Id'].nunique()}")
print(f"Tiempo de la etapa de datos: {data_seconds:.1f} s")
stage_times["01_prepare_data"] = data_seconds

2026-11-04 16:39:24,523 INFO mibici_mlops: Ejecución de la Fase 1 utilizada: mibici_20261026T025817_983267Z
2026-11-04 16:39:24,722 INFO mibici_mlops: Dataset integrado reutilizado: /home/josel/mlops/equipo-8-mibici-mlops/data/processed/mibici_integrated.parquet


,test,passed
0,T1 - Esquema,True
1,T2 - Tipos,True
2,T3 - Textos vacíos,True
3,T4 - Valores inválidos,True
4,T5 - Nulos,True
5,T6 - Rangos,True
6,T7 - Fechas consistentes,True
7,T8 - Duplicados,True
8,T9 - Objetivo,True


Dimensiones: (799264, 14)
Periodo: 2020-01-01 a 2026-09-30
Estaciones: 472
Tiempo de la etapa de datos: 0.4 s


### 6.2 Registro de versiones de las fuentes CSV
El inventario se ejecuta después de preparar las fuentes, de modo que incluye los CSV de clima, zonas y POI aunque hayan sido generados en esta corrida. El tiempo de cálculo de hashes se mide por separado.

In [12]:
start = time.perf_counter()
csv_inventory = version_csv_sources(DATA_DIR, CSV_VERSION_DIR)
stage_times["02_csv_source_versioning"] = time.perf_counter() - start
print(f"CSV inventariados: {len(csv_inventory)}")
display(csv_inventory[["file", "version", "known_versions"]].tail(12))

CSV inventariados: 92


,file,version,known_versions
80,data/raw/2026/datos_abiertos_2026_03.csv,1,1
81,data/raw/2026/datos_abiertos_2026_04.csv,1,1
82,data/raw/2026/datos_abiertos_2026_05.csv,1,1
83,data/raw/2026/datos_abiertos_2026_06.csv,1,1
84,data/raw/2026/datos_abiertos_2026_07.csv,1,1
85,data/raw/2026/datos_abiertos_2026_08.csv,1,1
86,data/raw/2026/datos_abiertos_2026_09.csv,1,1
87,data/raw/Manual_datos/nomenclatura_2026_09.csv,1,1
88,data/raw/pois/mibici_station_pois.csv,1,1
89,data/raw/weather/mibici_station_weather_zones.csv,1,1


### 6.3 Versiones de los archivos utilizados

El inventario CSV de `data/` se conserva, incluso para archivos nuevos. También
registramos SHA-256 y versiones de los Parquet, fuentes externas y archivos de la
corrida de Fase 1. Cada registro indica la ruta, el tipo de archivo y su uso.
No se suben archivos grandes a MLflow: se guardan sus identificadores y hashes.
La presencia de un CSV en `data/` no significa que se haya usado para entrenar.


In [13]:
# Archivos de entrada y procedencia de los datos.
PROVENANCE_DIR = PROJECT_ROOT / "reports" / "mlflow_provenance"
PROVENANCE_DIR.mkdir(parents=True, exist_ok=True)


def project_file_record(path: Path, role: str, used: bool = True) -> dict:
    """Describir un archivo sin copiarlo ni modificarlo."""
    path = Path(path)
    relative = path.relative_to(PROJECT_ROOT).as_posix() if path.is_relative_to(PROJECT_ROOT) else str(path)
    record = {"path": relative, "role": role, "used": used, "exists": path.is_file()}
    if path.is_file():
        record["size_bytes"] = path.stat().st_size
        record["sha256"] = calculate_sha256(path)
    return record


def build_data_provenance(inventory: pd.DataFrame) -> dict:
    """Relacionar las entradas conocidas con el integrado y su origen."""
    phase1_run = find_latest_valid_run(VALIDATION_REPORTS_DIR, len(EXPECTED_MONTHS))
    if phase1_run is None:
        raise FileNotFoundError("No se encontró la ejecución completa de la Fase 1.")
    used_sources = [
        (STATIONS_PATH, "catalogo_estaciones"),
        (WEATHER_PATH, "clima_exploratorio_y_zonas"),
        (STATION_ZONES_PATH, "zonas_estaticas"),
        (POI_PATH, "lugares_de_interes"),
        (phase1_run / "hourly_demand.parquet", "demanda_horaria_fase1"),
        (phase1_run / "manifest.json", "manifiesto_fase1"),
        (INTEGRATED_PATH, "dataset_integrado"),
    ]
    if DAILY_PATH.is_file():
        used_sources.append((DAILY_PATH, "demanda_diaria_intermedia"))
    if OSM_PATH.is_file():
        used_sources.append((OSM_PATH, "fuente_osm_para_poi_si_se_reconstruye"))
    files = [project_file_record(path, role, used=not role.startswith("fuente_osm_para_poi")) for path, role in used_sources]
    # Los CSV mensuales fueron procesados en la Fase 1; el manifiesto indica la corrida usada.
    monthly = inventory[inventory["file"].str.contains("datos_abiertos_", regex=False)]
    for row in monthly.itertuples(index=False):
        files.append({
            "path": row.file, "role": "csv_mensual_inventariado_fase1",
            "used": "upstream_no_verificado", "exists": True,
            "sha256": row.sha256, "size_bytes": int(row.size_bytes),
            "version": int(row.version),
        })
    by_path = {row.file: row for row in inventory.itertuples(index=False)}
    for file in files:
        if file["path"] in by_path:
            row = by_path[file["path"]]
            file["version"] = int(row.version)
            file["sha256"] = row.sha256
    return {
        "description": "Entradas conocidas y procedencia del dataset integrado de MiBici",
        "phase1_run": phase1_run.relative_to(PROJECT_ROOT).as_posix(),
        "integrated_sha256": calculate_sha256(INTEGRATED_PATH),
        "inventory_csv_count": len(inventory),
        "notes": [
            "El inventario completo contiene CSV encontrados, no necesariamente utilizados.",
            "Los CSV mensuales son candidatos de origen; la corrida procesada de Fase 1 es la entrada comprobada.",
            "Los archivos Parquet grandes no se duplican en MLflow.",
            "Los CSV externos pueden haberse reutilizado desde cache, sin descargarse en esta corrida.",
        ],
        "files": files,
    }


FILE_VERSION_DIR = PROJECT_ROOT / "reports" / "data_versioning" / "all_files"
FILE_VERSION_INDEX = FILE_VERSION_DIR / "file_versions.json"
FILE_VERSION_TABLE = FILE_VERSION_DIR / "file_versions.csv"


def version_used_files(sources: list[tuple[Path, str, str]]) -> pd.DataFrame:
    """Versionar archivos usados o generados; conservar el historial de cada ruta."""
    FILE_VERSION_DIR.mkdir(parents=True, exist_ok=True)
    previous = (
        json.loads(FILE_VERSION_INDEX.read_text(encoding="utf-8"))
        if FILE_VERSION_INDEX.is_file() else {}
    )
    records = dict(previous)
    rows = []
    for path, role, stage in sources:
        path = Path(path)
        relative = (
            path.relative_to(PROJECT_ROOT).as_posix()
            if path.is_relative_to(PROJECT_ROOT) else str(path)
        )
        if not path.is_file():
            raise FileNotFoundError(f"Archivo requerido no encontrado: {relative}")
        stat = path.stat()
        old = records.get(relative, {})
        # El hash solo se reutiliza si no cambiaron tamaño ni fecha de modificación.
        if old.get("size_bytes") == stat.st_size and old.get("mtime_ns") == stat.st_mtime_ns:
            sha256 = old["sha256"]
        else:
            sha256 = calculate_sha256(path)
        versions = list(old.get("versions", []))
        if sha256 not in versions:
            versions.append(sha256)
        version = versions.index(sha256) + 1
        records[relative] = {
            "sha256": sha256,
            "size_bytes": stat.st_size,
            "mtime_ns": stat.st_mtime_ns,
            "versions": versions,
            "version": version,
            "role": role,
            "stage": stage,
            "status": "present",
        }
        rows.append({
            "file": relative, "role": role, "stage": stage,
            "sha256": sha256, "version": version, "size_bytes": stat.st_size,
        })
    FILE_VERSION_INDEX.write_text(
        json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    # Conservar también los archivos registrados en llamadas anteriores.
    all_rows = [
        {"file": name, "role": item["role"], "stage": item["stage"],
         "sha256": item["sha256"], "version": item["version"],
         "size_bytes": item["size_bytes"]}
        for name, item in records.items()
        if item.get("status") == "present"
    ]
    table = pd.DataFrame(all_rows, columns=[
        "file", "role", "stage", "sha256", "version", "size_bytes"
    ]).drop_duplicates(subset=["file"])
    table.to_csv(FILE_VERSION_TABLE, index=False)
    return table


def collect_data_files() -> list[tuple[Path, str, str]]:
    """Reunir archivos del pipeline y fuentes de la Fase 1 sin asumir uso no demostrado."""
    files = []
    # Inventario completo de los CSV presentes en data, incluso los nuevos.
    for path in DATA_DIR.rglob("*"):
        if path.is_file() and path.suffix.lower() == ".csv":
            files.append((path, "csv_inventariado", "inventario_datos"))
    # Fuentes e intermedios de la ejecución actual.
    for path, role in [
        (STATIONS_PATH, "catalogo_estaciones"),
        (WEATHER_PATH, "clima_exploratorio"),
        (STATION_ZONES_PATH, "zonas_estaticas"),
        (POI_PATH, "poi"),
        (DAILY_PATH, "demanda_diaria"),
        (INTEGRATED_PATH, "dataset_integrado"),
    ]:
        if path.is_file():
            files.append((path, role, "preparacion_datos"))
    phase1 = find_latest_valid_run(VALIDATION_REPORTS_DIR, len(EXPECTED_MONTHS))
    if phase1 is not None:
        for path in phase1.rglob("*"):
            if path.is_file():
                files.append((path, "salida_fase_1", "fase_1"))
    # El PBF se documenta como fuente disponible; no se declara usado si solo se reutilizó POI.
    if OSM_PATH.is_file():
        files.append((OSM_PATH, "osm_fuente_disponible", "fuente_externa"))
    return files


# Guardar el inventario de archivos no CSV antes de registrar experimentos.
file_inventory = version_used_files(collect_data_files())
print(f"Archivos documentados con SHA-256: {file_inventory['file'].nunique()}")


data_provenance = build_data_provenance(csv_inventory)
provenance_path = PROVENANCE_DIR / "data_provenance.json"
provenance_path.write_text(json.dumps(data_provenance, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"CSV inventariados: {len(csv_inventory)}")
print(f"Entradas y archivos de procedencia documentados: {len(data_provenance['files'])}")
print(f"Manifiesto: {provenance_path.relative_to(PROJECT_ROOT)}")

Archivos documentados con SHA-256: 215
CSV inventariados: 92
Entradas y archivos de procedencia documentados: 90
Manifiesto: reports/mlflow_provenance/data_provenance.json


## 7. Variables del modelo (`features.py`)
El `features.py` actual es la plantilla vacía de Cookiecutter. Lo reemplazamos con las funciones de la sección 11.2 del 05.

- `add_history_features`: es la misma función del 05. Arma una tabla fecha × estación con **todos** los días del calendario (`asfreq("D")`), así los días sin registro quedan como `NaN` y no como cero. Sobre esa tabla calcula los rezagos (`lag_1`, `lag_7`, `lag_14`) y, con los datos desplazados un horizonte, los promedios móviles y los días activos de 7 y 28 días. Al inicio revisa que ningún rezago sea menor al horizonte, porque eso usaría información futura. El único cambio es que la columna objetivo ahora es un parámetro en vez de una variable global.
- `add_calendar_features`: agrega el día del mes y el año. El año no entra al modelo principal, pero se deja en la tabla como en el 05.
- `build_model_dataset`: ordena por estación y fecha y aplica las dos funciones anteriores. El orden de filas importa para que los resultados sean idénticos al 05.
- `split_by_date`: separa desarrollo (antes de 2026) y prueba (2026).

In [14]:
def add_history_features(data, horizon, lags, windows, target_column="target"):
    # Rezagos y promedios móviles calculados por fecha, no por fila.
    invalid_lags = [lag for lag in lags if lag < horizon]
    if invalid_lags:
        raise ValueError(
            f"Los rezagos {invalid_lags} usarían información futura con horizonte {horizon}."
        )
    if data.duplicated(["Origen_Id", "Fecha"]).any():
        raise ValueError("Hay fechas duplicadas por estación; no se pueden crear rezagos.")
    if not lags or not windows or any(window <= 0 for window in windows):
        raise ValueError("Los rezagos y ventanas deben ser listas no vacías de enteros positivos.")
    demand_grid = data.pivot(index="Fecha", columns="Origen_Id", values=target_column).asfreq("D")
    available = demand_grid.shift(horizon)

    history = {f"lag_{lag}": demand_grid.shift(lag) for lag in lags}
    for window in windows:
        history[f"roll_mean_{window}"] = available.rolling(window, min_periods=1).mean()
        history[f"active_days_{window}"] = available.rolling(window, min_periods=1).count()

    date_positions = demand_grid.index.get_indexer(data["Fecha"])
    station_positions = demand_grid.columns.get_indexer(data["Origen_Id"])
    result = data.copy()
    for name, values in history.items():
        result[name] = values.to_numpy()[date_positions, station_positions]
    return result


def add_calendar_features(data: pd.DataFrame) -> pd.DataFrame:
    # Componentes de la fecha que se conocen antes de predecir.
    result = data.copy()
    result["day_of_month"] = result["Fecha"].dt.day
    result["year"] = result["Fecha"].dt.year
    return result


def build_model_dataset(data, horizon, lags, windows, target_column="target"):
    # Tabla de modelado: orden estable, historia y calendario.
    ordered = data.sort_values(["Origen_Id", "Fecha"], kind="stable").reset_index(drop=True)
    with_history = add_history_features(ordered, horizon, lags, windows, target_column)
    return add_calendar_features(with_history)


def split_by_date(data: pd.DataFrame, test_start: pd.Timestamp):
    # Desarrollo antes de la fecha de corte; prueba desde ella.
    development = data.loc[data["Fecha"] < test_start].reset_index(drop=True)
    test = data.loc[data["Fecha"] >= test_start].reset_index(drop=True)
    return development, test

### 7.1 Construcción de la tabla de modelado
Generamos las variables, revisamos qué proporción falta en cada rezago (estaciones nuevas o días sin registro) y separamos desarrollo y prueba.

In [15]:
start = time.perf_counter()
model_dataset = build_model_dataset(integrated_data, HORIZON_DAYS, LAGS, ROLLING_WINDOWS, TARGET_COLUMN)
development_data, test_data = split_by_date(model_dataset, TEST_START)

print("Proporción de valores faltantes en la historia:")
print(model_dataset[HISTORY_COLUMNS].isna().mean().round(3).to_string())
for name, part in [("Desarrollo", development_data), ("Prueba", test_data)]:
    print(
        f"\n{name}: {len(part):,} registros, "
        f"{part['Fecha'].min().date()} a {part['Fecha'].max().date()}, "
        f"{part['Origen_Id'].nunique()} estaciones"
    )
stage_times["02_feature_engineering_and_split"] = time.perf_counter() - start

Proporción de valores faltantes en la historia:
lag_1             0.004
lag_7             0.008
lag_14            0.013
roll_mean_7       0.001
active_days_7     0.000
roll_mean_28      0.001
active_days_28    0.000

Desarrollo: 690,970 registros, 2020-01-01 a 2025-12-31, 374 estaciones

Prueba: 108,294 registros, 2026-01-01 a 2026-09-30, 469 estaciones


## 8. Validación temporal, métricas y líneas base (`modeling/evaluate.py`)
Módulo nuevo con las secciones 11.4, 11.6 y 11.9 del 05. En el 05 el ciclo de las ventanas estaba escrito dos veces (líneas base y modelos) y dependía de variables globales (`development_data`, `validation_splits`). Aquí cada función recibe los datos que usa, así se puede llamar desde `main.py` o desde una prueba.

- `walk_forward_splits`: genera los 12 cortes mensuales de 2025. En cada uno, el entrenamiento es todo lo anterior al corte (menos `HORIZON_DAYS - 1` días de hueco) y la validación es el mes siguiente. Devuelve las posiciones de las filas, que es el formato que acepta `GridSearchCV`.
- `describe_splits`: tabla con las fechas, filas y estaciones nuevas de cada ventana.
- `regression_metrics`: MAE y RMSE.
- Las tres líneas base del 05: ingenuo estacional (`lag_7`, y si falta, el promedio de la estación ese día de la semana), promedio estación-día y promedio global por día.
- `evaluate_baselines` y `evaluate_walk_forward`: recorren las ventanas y devuelven una fila por ventana. A los modelos ahora también les medimos el tiempo de entrenamiento, que se registra en MLflow.
- `add_skill` y `summarize_results`: mejora sobre el ingenuo en cada ventana y resumen con promedio y desviación estándar.
- `evaluate_by_segment`: métricas de la prueba 2026 para todas las estaciones, las que tienen historia y las nuevas.

In [16]:
NAIVE_BASELINE = "Ingenuo estacional (lag_7)"


def walk_forward_splits(dates: pd.Series, first_cutoff, n_folds: int, horizon: int):
    # Cortes mensuales: entrenar hasta el corte y validar el mes siguiente.
    splits = []
    for cutoff in pd.date_range(first_cutoff, periods=n_folds, freq="MS"):
        train_end = cutoff - pd.Timedelta(days=horizon - 1)
        validation_end = cutoff + pd.DateOffset(months=1)
        train_positions = np.flatnonzero(dates < train_end)
        validation_positions = np.flatnonzero((dates >= cutoff) & (dates < validation_end))
        if not len(train_positions) or not len(validation_positions):
            raise ValueError(f"Ventana sin datos de entrenamiento o validación: {cutoff.date()}")
        splits.append((cutoff, train_positions, validation_positions))
    return splits


def describe_splits(data: pd.DataFrame, splits) -> pd.DataFrame:
    rows = []
    for cutoff, train_positions, validation_positions in splits:
        train_stations = set(data.loc[train_positions, "Origen_Id"])
        validation_stations = set(data.loc[validation_positions, "Origen_Id"])
        rows.append({
            "cutoff": cutoff.date(),
            "train_end": data.loc[train_positions, "Fecha"].max().date(),
            "train_rows": len(train_positions),
            "validation_rows": len(validation_positions),
            "new_stations": len(validation_stations - train_stations),
        })
    return pd.DataFrame(rows)


def regression_metrics(y_true, y_pred) -> dict:
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
    }


def predict_station_weekday(train_data, data, target_column="target"):
    # Promedio de la estación ese día de la semana; si no hay, promedio global del día.
    means = train_data.groupby(["Origen_Id", "day_of_week"])[target_column].mean()
    keys = pd.MultiIndex.from_frame(data[["Origen_Id", "day_of_week"]])
    predictions = pd.Series(means.reindex(keys).to_numpy(), index=data.index)
    weekday_means = train_data.groupby("day_of_week")[target_column].mean()
    return predictions.fillna(data["day_of_week"].map(weekday_means))


def predict_seasonal_naive(train_data, data, target_column="target"):
    # Mismo día de la semana anterior; si falta, promedio estación-día.
    fallback = predict_station_weekday(train_data, data, target_column)
    return data["lag_7"].fillna(fallback)


def predict_global_weekday(train_data, data, target_column="target"):
    weekday_means = train_data.groupby("day_of_week")[target_column].mean()
    return data["day_of_week"].map(weekday_means)


BASELINES = {
    NAIVE_BASELINE: predict_seasonal_naive,
    "Promedio estación-día": predict_station_weekday,
    "Promedio global por día (versión anterior)": predict_global_weekday,
}


def evaluate_baselines(data, splits, target_column="target") -> pd.DataFrame:
    rows = []
    for cutoff, train_positions, validation_positions in splits:
        train_data = data.loc[train_positions]
        validation_data = data.loc[validation_positions]
        for name, predict in BASELINES.items():
            predictions = predict(train_data, validation_data, target_column)
            rows.append({
                "cutoff": cutoff.date(),
                "model": name,
                **regression_metrics(validation_data[target_column], predictions),
            })
    return pd.DataFrame(rows)


def evaluate_walk_forward(pipeline, data, splits, columns, model_name, target_column="target"):
    # Reentrenar una copia del pipeline en cada ventana y medir el error.
    rows = []
    for cutoff, train_positions, validation_positions in splits:
        train_data = data.loc[train_positions]
        validation_data = data.loc[validation_positions]
        start = time.perf_counter()
        fitted = clone(pipeline).fit(train_data[columns], train_data[target_column])
        fit_seconds = time.perf_counter() - start
        predictions = fitted.predict(validation_data[columns])
        rows.append({
            "cutoff": cutoff.date(),
            "model": model_name,
            **regression_metrics(validation_data[target_column], predictions),
            "fit_seconds": fit_seconds,
            "train_rows": len(train_positions),
            "validation_rows": len(validation_positions),
        })
        logger.info("%s | %s | MAE %.3f", model_name, cutoff.date(), rows[-1]["mae"])
    return pd.DataFrame(rows)


def add_skill(results: pd.DataFrame, baseline_results: pd.DataFrame) -> pd.DataFrame:
    # Mejora sobre el ingenuo estacional en la misma ventana.
    naive_mae = baseline_results.loc[
        baseline_results["model"] == NAIVE_BASELINE
    ].set_index("cutoff")["mae"]
    results = results.copy()
    results["skill_vs_naive"] = 1 - results["mae"] / results["cutoff"].map(naive_mae)
    return results


def summarize_results(results: pd.DataFrame) -> pd.DataFrame:
    return (
        results.groupby("model")
        .agg(
            mae=("mae", "mean"),
            mae_std=("mae", "std"),
            rmse=("rmse", "mean"),
            skill_vs_naive=("skill_vs_naive", "mean"),
        )
        .sort_values("mae")
    )


def evaluate_by_segment(y_true, model_predictions, naive_predictions, segments) -> pd.DataFrame:
    # Métricas de prueba para todas las estaciones, con historia y nuevas.
    rows = []
    for segment in ["Todas", "Con historia", "Nueva"]:
        mask = np.ones(len(y_true), dtype=bool) if segment == "Todas" else segments == segment
        if not mask.any():
            continue
        model_metrics = regression_metrics(y_true[mask], model_predictions[mask])
        naive_metrics = regression_metrics(y_true[mask], naive_predictions[mask])
        rows.append({
            "stations": segment,
            "rows": int(mask.sum()),
            "model_mae": model_metrics["mae"],
            "model_rmse": model_metrics["rmse"],
            "naive_mae": naive_metrics["mae"],
            "skill_vs_naive": 1 - model_metrics["mae"] / naive_metrics["mae"],
        })
    return pd.DataFrame(rows)

### 8.1 Ventanas de validación
Generamos las 12 ventanas sobre el conjunto de desarrollo. `cv_splits` es la misma información sin la fecha de corte, en el formato que espera `GridSearchCV`.

In [17]:
start = time.perf_counter()
validation_splits = walk_forward_splits(development_data["Fecha"], FIRST_CUTOFF, N_FOLDS, HORIZON_DAYS)
cv_splits = [(train, validation) for _, train, validation in validation_splits]
display(describe_splits(development_data, validation_splits))
stage_times["03_validation_splits"] = time.perf_counter() - start

,cutoff,train_end,train_rows,validation_rows,new_stations
0,2025-01-01,2024-12-31,557255,11373,0
1,2025-02-01,2025-01-31,568628,10230,0
2,2025-03-01,2025-02-28,578858,11355,0
3,2025-04-01,2025-03-31,590213,10991,0
4,2025-05-01,2025-04-30,601204,11366,0
5,2025-06-01,2025-05-31,612570,10978,1
6,2025-07-01,2025-06-30,623548,11352,0
7,2025-08-01,2025-07-31,634900,11363,0
8,2025-09-01,2025-08-31,646263,11000,0
9,2025-10-01,2025-09-30,657263,11363,1


### 8.2 Líneas base
Evaluamos las tres líneas base en las 12 ventanas. Los promedios se calculan solo con el entrenamiento de cada ventana.

In [18]:
start = time.perf_counter()
baseline_results = evaluate_baselines(development_data, validation_splits, TARGET_COLUMN)
baseline_results = add_skill(baseline_results, baseline_results)
baseline_seconds = time.perf_counter() - start
display(summarize_results(baseline_results).round(3))
stage_times["04_baselines"] = baseline_seconds

,mae,mae_std,rmse,skill_vs_naive
model,,,,
Ingenuo estacional (lag_7),7.224,0.689,11.832,0.000
Promedio estación-día,7.703,0.776,12.678,-0.070
Promedio global por día (versión anterior),22.986,0.685,33.777,-2.207


## 9. Pipeline y entrenamiento (`modeling/train.py`)
El `train.py` actual también es la plantilla de Cookiecutter. Lo reemplazamos con las secciones 11.7, 11.11 y 11.13 del 05.

- `build_model_pipeline`: arma el `ColumnTransformer` con la tabla de tratamiento de variables y le pega el modelo. Decide qué paso aplica a cada columna, así sirve para cualquier conjunto de variables:
  - `Origen_Id` → `TargetEncoder` (nivel de demanda de cada estación, aprendido solo con el entrenamiento).
  - `day_of_week`, `month`, `weather_zone` → `OneHotEncoder` que ignora categorías nuevas.
  - Rezagos y promedios → `SimpleImputer` con la mediana y una columna que indica si faltaba.
  - Lo demás (día del mes, POI y, si se usa, clima) → sin cambios.
- `make_model`: crea Random Forest o HistGradientBoosting combinando los parámetros iniciales con los que se le pasen.
- `run_grid_search`: `GridSearchCV` con las ventanas del walk-forward. Usa una de cada dos ventanas para que tarde la mitad.
- `tuning_table` y `best_configuration`: ordenan las combinaciones por MAE y devuelven la mejor.
- `train_final_model` y `save_model`: entrenan con todo el desarrollo (2020–2025) y guardan el pipeline en `models/`.

In [19]:
def build_model_pipeline(
    model,
    columns,
    station_id_columns=STATION_ID_COLUMNS,
    onehot_columns=ONEHOT_COLUMNS,
    history_columns=HISTORY_COLUMNS,
    seed=SEED,
) -> Pipeline:
    # Preprocesamiento según la tabla de tratamiento de variables, más el modelo.
    station = [c for c in columns if c in station_id_columns]
    categorical = [c for c in columns if c in onehot_columns]
    history = [c for c in columns if c in history_columns]
    numeric = [c for c in columns if c not in station + categorical + history]

    transformers = []
    if station:
        transformers.append(
            ("station", TargetEncoder(target_type="continuous", random_state=seed), station)
        )
    if categorical:
        transformers.append((
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical,
        ))
    if history:
        transformers.append((
            "history",
            SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True),
            history,
        ))
    if numeric:
        transformers.append(("numeric", "passthrough", numeric))
    preprocessing = ColumnTransformer(transformers=transformers, remainder="drop")
    return Pipeline(steps=[("preprocessing", preprocessing), ("model", model)])


def make_model(model_name: str, params: dict | None = None):
    if model_name == "Random Forest":
        return RandomForestRegressor(**{**RF_INITIAL_PARAMS, **(params or {})})
    if model_name == "HistGradientBoosting":
        return HistGradientBoostingRegressor(**{**HGB_INITIAL_PARAMS, **(params or {})})
    raise ValueError(f"Modelo desconocido: {model_name}")


def run_grid_search(model_name, grid, data, columns, splits, target_column="target"):
    # Búsqueda de hiperparámetros con las ventanas temporales; 2026 no participa.
    search = GridSearchCV(
        estimator=build_model_pipeline(make_model(model_name), columns),
        param_grid=grid,
        scoring={"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"},
        refit=False,
        cv=splits,
        n_jobs=1,
        verbose=1,
    )
    return search.fit(data[columns], data[target_column])


def tuning_table(searches: dict) -> pd.DataFrame:
    tables = []
    for model_name, search in searches.items():
        results = pd.DataFrame(search.cv_results_)
        results["model"] = model_name
        results["mae"] = -results["mean_test_mae"]
        results["mae_std"] = results["std_test_mae"]
        results["rmse"] = -results["mean_test_rmse"]
        tables.append(results[["model", "params", "mae", "mae_std", "rmse"]])
    return pd.concat(tables, ignore_index=True).sort_values("mae", ignore_index=True)


def best_configuration(tuning_results: pd.DataFrame | None) -> tuple[str, dict]:
    # Mejor combinación de la búsqueda o, sin búsqueda, la guardada en config.
    if tuning_results is None or tuning_results.empty:
        return FINAL_MODEL, BEST_PARAMS[FINAL_MODEL]
    best = tuning_results.iloc[0]
    params = {key.replace("model__", ""): value for key, value in best["params"].items()}
    return best["model"], params


def train_final_model(model_name, params, data, columns, target_column="target") -> Pipeline:
    pipeline = build_model_pipeline(make_model(model_name, params), columns)
    return pipeline.fit(data[columns], data[target_column])


def save_model(pipeline: Pipeline, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipeline, path)
    return path

### 9.1 Pipeline de ejemplo
Ajustamos el pipeline con 5,000 filas solo para ver su estructura y cuántas columnas recibe el modelo después del preprocesamiento.

In [20]:
start = time.perf_counter()
example_pipeline = build_model_pipeline(make_model("Random Forest"), FEATURE_COLUMNS)
example_pipeline.fit(
    development_data[FEATURE_COLUMNS].head(5000), development_data[TARGET_COLUMN].head(5000)
)
print(example_pipeline)
print("Columnas después del preprocesamiento:",
      len(example_pipeline.named_steps["preprocessing"].get_feature_names_out()))
stage_times["05_pipeline_example"] = time.perf_counter() - start

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('station',
                                                  TargetEncoder(random_state=42,
                                                                target_type='continuous'),
                                                  ['Origen_Id']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['day_of_week', 'month',
                                                   'weather_zone']),
                                                 ('history',
                                                  SimpleImputer(add_indicator=True,
                                                                keep_empty_features=True,
                                                

### 9.2 Modelos con parámetros iniciales
Evaluamos Random Forest e HistGradientBoosting en las 12 ventanas con el conjunto principal. Si `COMPARE_FEATURE_SETS = True`, también se evalúan los otros dos conjuntos sin clima, como en la 11.8 del 05.

In [21]:
feature_sets_to_compare = (
    ["original_11_variables", "calendario_estacion", MAIN_FEATURE_SET]
    if COMPARE_FEATURE_SETS else [MAIN_FEATURE_SET]
)
model_runs = {}
for feature_set in feature_sets_to_compare:
    columns = FEATURE_SETS[feature_set]
    for model_name in MODEL_NAMES:
        label = f"{model_name} | {feature_set}"
        pipeline = build_model_pipeline(make_model(model_name), columns)
        model_start = time.perf_counter()
        model_runs[label] = {
            "model_name": model_name,
            "feature_set": feature_set,
            "params": pipeline.named_steps["model"].get_params(),
            "results": add_skill(
                evaluate_walk_forward(
                    pipeline, development_data, validation_splits, columns, label, TARGET_COLUMN
                ),
                baseline_results,
            ),
        }

        stage_times[f"06_cv_{model_name}_{feature_set}"] = time.perf_counter() - model_start

model_results = pd.concat([run["results"] for run in model_runs.values()], ignore_index=True)
all_results = pd.concat([baseline_results, model_results], ignore_index=True)
model_comparison = summarize_results(all_results)
display(model_comparison.round(3))

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
2026-11-04 16:40:35,622 INFO mibici_mlops: Random Forest | calendario_estacion_historia | 2025-01-01 | MAE 5.443
/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
2026-11-04 16:41:32,442 INFO mibici_mlops: Random Forest | calendario_estacion_historia | 2025-02-01 | MAE 5.787
/home/josel/mlops/equipo-8-mib

,mae,mae_std,rmse,skill_vs_naive
model,,,,
HistGradientBoosting | calendario_estacion_historia,5.440,0.393,8.653,0.244
Random Forest | calendario_estacion_historia,5.456,0.383,8.800,0.242
Ingenuo estacional (lag_7),7.224,0.689,11.832,0.000
Promedio estación-día,7.703,0.776,12.678,-0.070
Promedio global por día (versión anterior),22.986,0.685,33.777,-2.207


### 9.3 Clima observado (comparación retrospectiva, opcional)
Igual que en la 11.10 del 05: el modelo con menor MAE se vuelve a evaluar agregando la temperatura y la precipitación observadas. Es solo una comparación; el modelo final no usa clima porque al predecir no se conoce el clima real del día siguiente.

In [22]:
start = time.perf_counter()
main_models = model_comparison.loc[model_comparison.index.str.endswith(f"| {MAIN_FEATURE_SET}")]
best_initial_model = main_models.index[0].split(" | ")[0]
print(f"Mejor modelo con el conjunto principal: {best_initial_model}")

if RUN_WEATHER_REFERENCE:
    weather_set = "historia_y_clima_observado"
    columns = FEATURE_SETS[weather_set]
    label = f"{best_initial_model} | {weather_set}"
    pipeline = build_model_pipeline(make_model(best_initial_model), columns)
    model_runs[label] = {
        "model_name": best_initial_model,
        "feature_set": weather_set,
        "params": pipeline.named_steps["model"].get_params(),
        "results": add_skill(
            evaluate_walk_forward(
                pipeline, development_data, validation_splits, columns, label, TARGET_COLUMN
            ),
            baseline_results,
        ),
    }
    display(summarize_results(pd.concat(
        [model_runs[f"{best_initial_model} | {MAIN_FEATURE_SET}"]["results"],
         model_runs[label]["results"]]
    )).round(3))
else:
    print("Comparación omitida (RUN_WEATHER_REFERENCE = False).")
stage_times["07_weather_reference"] = time.perf_counter() - start

Mejor modelo con el conjunto principal: HistGradientBoosting
Comparación omitida (RUN_WEATHER_REFERENCE = False).


### 9.4 Búsqueda de hiperparámetros (opcional)
Con `RUN_GRID_SEARCH = False` se usan los parámetros de `BEST_PARAMS`. Las rejillas quedan en `config.py` como evidencia de lo que se probó. Si se enciende, la mejor combinación se usa directamente en el modelo final y se imprime para copiarla a `BEST_PARAMS`.

In [23]:
start = time.perf_counter()
grid_searches = {}
tuning_results = None
if RUN_GRID_SEARCH:
    tuning_splits = cv_splits[::TUNING_FOLD_STEP]
    for model_name, grid in PARAMETER_GRIDS.items():
        grid_searches[model_name] = run_grid_search(
            model_name, grid, development_data, FEATURE_COLUMNS, tuning_splits, TARGET_COLUMN
        )
    tuning_results = tuning_table(grid_searches)
    display(tuning_results.head(10))
else:
    print("Búsqueda omitida (RUN_GRID_SEARCH = False). Parámetros guardados:")
    for model_name, params in BEST_PARAMS.items():
        print(f"  {model_name}: {params}")

final_model_name, final_params = best_configuration(tuning_results)
print(f"\nModelo final: {final_model_name}")
print(f"Hiperparámetros: {final_params}")
stage_times["08_hyperparameter_search"] = time.perf_counter() - start

Búsqueda omitida (RUN_GRID_SEARCH = False). Parámetros guardados:
  Random Forest: {'n_estimators': 150, 'max_depth': 15, 'min_samples_leaf': 5, 'max_features': 0.7}
  HistGradientBoosting: {'learning_rate': 0.1, 'max_iter': 200, 'max_leaf_nodes': 31, 'loss': 'poisson'}

Modelo final: Random Forest
Hiperparámetros: {'n_estimators': 150, 'max_depth': 15, 'min_samples_leaf': 5, 'max_features': 0.7}


### 9.5 Modelo final: validación en 2025 y prueba en 2026
Primero comprobamos la configuración final en las 12 ventanas. Después la entrenamos una sola vez con todo 2020–2025 y evaluamos 2026, junto con el ingenuo estacional en las mismas filas. Los resultados de 2026 no se usan para cambiar nada del modelo.

In [24]:
start = time.perf_counter()
final_label = f"{final_model_name} | final"
final_walk_forward = add_skill(
    evaluate_walk_forward(
        build_model_pipeline(make_model(final_model_name, final_params), FEATURE_COLUMNS),
        development_data, validation_splits, FEATURE_COLUMNS, final_label, TARGET_COLUMN,
    ),
    baseline_results,
)
display(final_walk_forward.round(3))
naive_validation_mae = baseline_results.loc[baseline_results["model"] == NAIVE_BASELINE, "mae"].mean()
print(f"MAE promedio: {final_walk_forward['mae'].mean():.3f} (ingenuo estacional: {naive_validation_mae:.3f})")
stage_times["09_final_walk_forward"] = time.perf_counter() - start

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
2026-11-04 16:54:41,682 INFO mibici_mlops: Random Forest | final | 2025-01-01 | MAE 5.439
/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
2026-11-04 16:55:42,012 INFO mibici_mlops: Random Forest | final | 2025-02-01 | MAE 5.760
/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/s

,cutoff,model,mae,rmse,fit_seconds,train_rows,validation_rows,skill_vs_naive
0,2025-01-01,Random Forest | final,5.439,8.322,54.435,557255,11373,0.268
1,2025-02-01,Random Forest | final,5.760,9.476,58.319,568628,10230,0.260
2,2025-03-01,Random Forest | final,5.458,8.376,60.841,578858,11355,0.224
3,2025-04-01,Random Forest | final,5.755,8.939,62.202,590213,10991,0.297
4,2025-05-01,Random Forest | final,5.438,8.395,62.531,601204,11366,0.214
5,2025-06-01,Random Forest | final,5.538,8.328,64.836,612570,10978,0.224
6,2025-07-01,Random Forest | final,4.785,7.201,67.799,623548,11352,0.235
7,2025-08-01,Random Forest | final,4.916,7.268,68.233,634900,11363,0.213
8,2025-09-01,Random Forest | final,6.152,12.254,71.890,646263,11000,0.248
9,2025-10-01,Random Forest | final,5.107,7.616,74.082,657263,11363,0.204


MAE promedio: 5.438 (ingenuo estacional: 7.224)


In [25]:
start = time.perf_counter()
final_pipeline = train_final_model(
    final_model_name, final_params, development_data, FEATURE_COLUMNS, TARGET_COLUMN
)
final_fit_seconds = time.perf_counter() - start
save_model(final_pipeline, FINAL_MODEL_PATH)

prediction_start = time.perf_counter()
final_predictions = final_pipeline.predict(test_data[FEATURE_COLUMNS])
stage_times["11_test_prediction"] = time.perf_counter() - prediction_start
naive_predictions = predict_seasonal_naive(development_data, test_data, TARGET_COLUMN).to_numpy()
known_stations = set(development_data["Origen_Id"])
test_segments = np.where(test_data["Origen_Id"].isin(known_stations), "Con historia", "Nueva")

test_results = evaluate_by_segment(
    test_data[TARGET_COLUMN].to_numpy(), final_predictions, naive_predictions, test_segments
)
display(test_results.round(3))
print(f"Entrenamiento final: {final_fit_seconds:.1f} s | Modelo guardado en {FINAL_MODEL_PATH}")
stage_times["10_final_fit"] = final_fit_seconds

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


,stations,rows,model_mae,model_rmse,naive_mae,skill_vs_naive
0,Todas,108294,5.896,10.782,7.957,0.259
1,Con historia,99487,6.108,11.142,8.101,0.246
2,Nueva,8807,3.511,5.206,6.330,0.445


Entrenamiento final: 83.8 s | Modelo guardado en /home/josel/mlops/equipo-8-mibici-mlops/models/mibici_final_model.joblib


## 10. Gráficas del modelado (`plots.py`)
Las funciones actuales de `plots.py` y `reporting.py` pueden abrir ventanas con `plt.show()`. En un notebook eso solo muestra la figura, pero desde la terminal abre una ventana y detiene la ejecución hasta cerrarla. Agregamos tres funciones que siempre guardan y cierran la figura:

- `save_figure`: crea la carpeta, guarda el PNG y cierra la figura para liberar memoria. Devuelve la ruta, que después registramos en MLflow.
- `plot_mae_by_fold`: MAE por ventana de las líneas base y los modelos (la gráfica de la 11.9 del 05).
- `plot_test_predictions`: viajes reales, predicción y ingenuo en 2026, para el total diario y para la estación con más demanda (la gráfica de la 11.14 del 05).

En el `finish_plot` actual de `plots.py` habría que hacer el mismo cambio: reemplazar `plt.show()` por `plt.close()` cuando se pasa `output_dir`.

In [26]:
def save_figure(fig, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    return path


def plot_mae_by_fold(results: pd.DataFrame, models: list[str], path: Path) -> Path:
    fig, ax = plt.subplots(figsize=(12, 5))
    for model_name in models:
        model_rows = results.loc[results["model"] == model_name]
        ax.plot(model_rows["cutoff"].astype(str), model_rows["mae"], marker="o", label=model_name)
    ax.set_title("MAE por ventana de validación (mes evaluado)")
    ax.set_xlabel("Inicio del mes evaluado")
    ax.set_ylabel("MAE (viajes por estación-día)")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)
    ax.legend()
    return save_figure(fig, path)


def plot_test_predictions(review: pd.DataFrame, target_column: str, path: Path) -> Path:
    columns = [target_column, "predicted_trips", "naive_prediction"]
    daily_totals = review.groupby("Fecha")[columns].sum()
    station = (
        review.loc[review["segment"] == "Con historia"]
        .groupby("Origen_Id")[target_column].mean().idxmax()
    )
    station_series = review.loc[review["Origen_Id"] == station].set_index("Fecha")[columns]
    styles = [
        (target_column, "Viajes reales", "black", 1.6),
        ("predicted_trips", "Predicción del modelo", "tab:blue", 1.2),
        ("naive_prediction", "Ingenuo estacional (lag_7)", "tab:orange", 1.0),
    ]
    fig, axes = plt.subplots(2, 1, figsize=(14, 9))
    for column, label, color, width in styles:
        axes[0].plot(daily_totals.index, daily_totals[column], label=label, color=color, linewidth=width)
        axes[1].plot(station_series.index, station_series[column], label=label, color=color, linewidth=width)
    axes[0].set_title("Viajes diarios totales en 2026 (suma de todas las estaciones)")
    axes[1].set_title(f"Estación {station} (mayor demanda media en 2026)")
    for ax in axes:
        ax.set_xlabel("Fecha")
        ax.set_ylabel("Viajes por día")
        ax.grid(alpha=0.3)
        ax.legend()
    return save_figure(fig, path)

### 10.1 Gráficas exploratorias de la Fase 1
Se vuelven a generar las gráficas exploratorias que recuperamos para el notebook 06. En MLflow se guardan bajo `eda/fase_1`, separadas por demanda, estaciones y fuentes externas. No son necesariamente los mismos archivos PNG de la entrega original; son gráficas generadas de nuevo con los datos del pipeline actual. El clima es solo exploratorio y no forma parte de las 18 variables del modelo final.

Las series numéricas de demanda por día de la semana y por mes también se guardan en **Model metrics**. Las gráficas completas siguen en **Artifacts**.


In [27]:
def plot_exploratory_figures(data: pd.DataFrame, poi_columns: list[str], output_dir: Path) -> dict:
    """Guardar las gráficas exploratorias del notebook 05 sin mostrar ventanas."""
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = {}

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(data["target"], bins=60, edgecolor="black")
    ax.set(title="Distribución de viajes diarios por estación", xlabel="Viajes diarios", ylabel="Frecuencia")
    saved["demand_distribution"] = save_figure(fig, output_dir / "demand_distribution.png")

    weekday = data.groupby("day_of_week")["target"].mean().reindex(range(7))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"], weekday)
    ax.set(title="Promedio de viajes por día de la semana", ylabel="Viajes por estación y día")
    saved["weekday_demand"] = save_figure(fig, output_dir / "weekday_demand.png")

    monthly = data.groupby("month")["target"].mean().reindex(range(1, 13))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(range(1, 13), monthly)
    ax.set(title="Promedio de viajes por mes", xlabel="Mes", ylabel="Viajes por estación y día")
    saved["monthly_demand"] = save_figure(fig, output_dir / "monthly_demand.png")

    station_mean = data.groupby("Origen_Id")["target"].mean().sort_values()
    fig, axes = plt.subplots(2, 1, figsize=(12, 9))
    station_mean.tail(10).plot.bar(ax=axes[0], title="10 estaciones con mayor demanda media")
    station_mean.head(10).plot.bar(ax=axes[1], title="10 estaciones con menor demanda media")
    for ax in axes:
        ax.set(xlabel="Estación", ylabel="Viajes por día")
    fig.tight_layout()
    saved["station_ranking"] = save_figure(fig, output_dir / "station_ranking.png")

    if set(poi_columns).issubset(data.columns):
        station_poi = data.drop_duplicates("Origen_Id").set_index("Origen_Id")[poi_columns]
        fig, axes = plt.subplots(2, 1, figsize=(13, 9))
        station_poi.reindex(station_mean.tail(10).index).plot.bar(stacked=True, ax=axes[0], title="POI en estaciones de mayor demanda")
        station_poi.reindex(station_mean.head(10).index).plot.bar(stacked=True, ax=axes[1], title="POI en estaciones de menor demanda")
        fig.tight_layout()
        saved["station_poi"] = save_figure(fig, output_dir / "station_poi.png")

    if {"temperature_2m_mean", "precipitation_sum"}.issubset(data.columns):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(data["temperature_2m_mean"].dropna(), bins=35)
        axes[0].set(title="Distribución de temperatura (EDA)", xlabel="Temperatura media (°C)")
        axes[1].hist(data["precipitation_sum"].dropna(), bins=35)
        axes[1].set(title="Distribución de precipitación (EDA)", xlabel="Precipitación (mm)")
        saved["weather_distribution"] = save_figure(fig, output_dir / "weather_distribution.png")

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        temp_bins = pd.cut(data["temperature_2m_mean"], bins=10)
        data.groupby(temp_bins, observed=True)["target"].mean().plot.bar(ax=axes[0])
        axes[0].set(title="Demanda media por temperatura (EDA)", xlabel="Rango de temperatura")
        rain_bins = [-0.001, 0, 1, 5, 20, float("inf")]
        rain_labels = ["Sin lluvia", "0-1", "1-5", "5-20", ">20"]
        rain = pd.cut(data["precipitation_sum"], bins=rain_bins, labels=rain_labels, include_lowest=True)
        data.groupby(rain, observed=True)["target"].mean().plot.bar(ax=axes[1])
        axes[1].set(title="Demanda media por lluvia (EDA)", xlabel="Precipitación (mm)")
        fig.tight_layout()
        saved["weather_vs_demand"] = save_figure(fig, output_dir / "weather_vs_demand.png")
    return saved

### 10.2 Gráficas complementarias de la Fase 2
Estas gráficas se agregaron durante el desarrollo del modelo: cobertura anual, periodos sin registros, días del mes, festivos y comparaciones de demanda con POI. Se guardan en `eda/fase_2`. Las gráficas de validación, comparación de modelos y prueba de 2026 permanecen en `modeling/`.

In [28]:
def plot_additional_eda(data: pd.DataFrame, poi_columns: list[str], output_dir: Path) -> dict:
    """Recuperar las visualizaciones de calidad y EDA del notebook 05."""
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = {}

    annual = data.assign(year=data["Fecha"].dt.year).groupby("year").agg(
        records=("target", "size"),
        stations=("Origen_Id", "nunique"),
        trips=("target", "sum"),
    )
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(annual.index.astype(str), annual["stations"])
    ax.set(title="Estaciones con registros por año", xlabel="Año", ylabel="Número de estaciones")
    saved["annual_station_coverage"] = save_figure(fig, output_dir / "annual_station_coverage.png")

    # Las fechas son únicas por estación y día según el contrato de datos.
    station_dates = data[["Origen_Id", "Fecha"]].sort_values(["Origen_Id", "Fecha"])
    previous = station_dates.groupby("Origen_Id")["Fecha"].shift()
    gap_days = (station_dates["Fecha"] - previous).dt.days.sub(1)
    gap_days = gap_days.loc[gap_days > 0]
    gap_groups = pd.cut(
        gap_days,
        bins=[0, 1, 7, 30, float("inf")],
        labels=["1 día", "2 a 7 días", "8 a 30 días", "Más de 30 días"],
    )
    gap_counts = gap_groups.value_counts(sort=False)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(gap_counts.index.astype(str), gap_counts.values)
    ax.set(title="Periodos consecutivos sin registros", xlabel="Duración", ylabel="Cantidad de periodos")
    saved["missing_periods"] = save_figure(fig, output_dir / "missing_periods.png")

    # Diferencia de cobertura por fecha sin expandir todos los intervalos faltantes.
    # Para cada estación se cuentan los días desde su primera hasta su última fecha.
    station_span = data.groupby("Origen_Id")["Fecha"].agg(["min", "max"])
    dates = pd.date_range(data["Fecha"].min(), data["Fecha"].max(), freq="D")
    starts = station_span["min"].value_counts().reindex(dates, fill_value=0).cumsum()
    ends = station_span["max"].value_counts().reindex(dates, fill_value=0).cumsum().shift(1, fill_value=0)
    expected = starts.sub(ends)
    observed = data.groupby("Fecha")["Origen_Id"].nunique().reindex(dates, fill_value=0)
    missing = expected.sub(observed).clip(lower=0)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(missing.index, missing.values)
    ax.set(title="Estaciones sin registros por fecha (dentro de su periodo observado)",
           xlabel="Fecha", ylabel="Estaciones sin registros")
    ax.grid(alpha=0.3)
    saved["shared_missing_dates"] = save_figure(fig, output_dir / "shared_missing_dates.png")

    day_of_month = data.groupby(data["Fecha"].dt.day)["target"].mean()
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(day_of_month.index, day_of_month.values, marker="o")
    ax.set(title="Demanda promedio según el día del mes", xlabel="Día del mes",
           ylabel="Viajes promedio por estación-día observado")
    ax.set_xticks(range(1, 32, 2))
    ax.grid(alpha=0.2)
    saved["day_of_month_demand"] = save_figure(fig, output_dir / "day_of_month_demand.png")

    # Evolución de la demanda por mes calendario (distinta del promedio por mes del año).
    monthly = data.groupby(data["Fecha"].dt.to_period("M"))["target"].mean()
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(monthly.index.to_timestamp(), monthly.values)
    ax.set(title="Evolución mensual de la demanda media por estación", xlabel="Mes",
           ylabel="Viajes promedio por estación-día observado")
    ax.grid(alpha=0.3)
    saved["monthly_demand_trend"] = save_figure(fig, output_dir / "monthly_demand_trend.png")

    if set(poi_columns).issubset(data.columns):
        station = data.groupby("Origen_Id")["target"].mean().rename("mean_target").to_frame()
        station_poi = data.drop_duplicates("Origen_Id").set_index("Origen_Id")[poi_columns]
        station = station.join(station_poi, how="inner")
        correlations = station[poi_columns].corrwith(station["mean_target"]).dropna().sort_values()
        if not correlations.empty:
            fig, ax = plt.subplots(figsize=(11, 5))
            ax.bar(correlations.index, correlations.values)
            ax.axhline(0, color="black", linewidth=0.8)
            ax.set(title="Correlación descriptiva entre POI y demanda media por estación",
                   xlabel="Categoría de POI", ylabel="Correlación de Pearson")
            ax.tick_params(axis="x", rotation=45)
            saved["poi_demand_correlation"] = save_figure(fig, output_dir / "poi_demand_correlation.png")

        for label, selected in (("top", station.nlargest(10, "mean_target")),
                                ("bottom", station.nsmallest(10, "mean_target"))):
            if selected.empty:
                continue
            totals = selected[poi_columns].sum(axis=1)
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].bar(selected.index.astype(str), selected["mean_target"])
            axes[0].set(title=f"Demanda media diaria: estaciones {label}", ylabel="Viajes diarios")
            axes[1].bar(selected.index.astype(str), totals)
            axes[1].set(title="Cantidad total de POI", xlabel="ID de estación", ylabel="POI en 500 m")
            saved[f"{label}_station_demand_vs_poi"] = save_figure(
                fig, output_dir / f"{label}_station_demand_vs_poi.png"
            )

        attribute_changes = data.groupby("Origen_Id")[["weather_zone"] + poi_columns].nunique()
        inconsistent = attribute_changes.gt(1).sum()
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.bar(inconsistent.index, inconsistent.values)
        ax.set(title="Estaciones con atributos de zona o POI variables en el tiempo",
               xlabel="Atributo", ylabel="Estaciones con cambios")
        ax.tick_params(axis="x", rotation=45)
        saved["station_attribute_consistency"] = save_figure(
            fig, output_dir / "station_attribute_consistency.png"
        )

    return saved


def plot_holiday_demand(data: pd.DataFrame, output_dir: Path) -> dict:
    """Reproducir el análisis descriptivo de días oficiales de descanso del notebook 05."""
    from datetime import timedelta

    def nth_monday(year: int, month: int, occurrence: int) -> date:
        first = date(year, month, 1)
        return first + timedelta(days=(7 - first.weekday()) % 7 + 7 * (occurrence - 1))

    records = []
    for year in range(int(data["Fecha"].dt.year.min()), int(data["Fecha"].dt.year.max()) + 1):
        holidays = [
            (date(year, 1, 1), "Año Nuevo"),
            (nth_monday(year, 2, 1), "Constitución"),
            (nth_monday(year, 3, 3), "Natalicio de Benito Juárez"),
            (date(year, 5, 1), "Día del Trabajo"),
            (date(year, 9, 16), "Independencia"),
            (nth_monday(year, 11, 3), "Revolución Mexicana"),
            (date(year, 12, 25), "Navidad"),
        ]
        if year == 2024:
            holidays.append((date(year, 10, 1), "Cambio de gobierno federal"))
        records.extend((pd.Timestamp(day), name) for day, name in holidays)

    holiday_dates = pd.DataFrame(records, columns=["Fecha", "holiday_name"])
    daily = data.groupby("Fecha")["target"].mean().rename("mean_demand").reset_index()
    holidays = daily.merge(holiday_dates, on="Fecha", how="inner", validate="one_to_one")
    summary = holidays.groupby("holiday_name")["mean_demand"].mean().sort_values()
    if summary.empty:
        return {}
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(summary.index, summary.values)
    ax.set(title="Demanda promedio de MiBici por festividad (2020–2026)",
           xlabel="Viajes diarios promedio por estación", ylabel="Festividad")
    return {"holiday_demand": save_figure(fig, output_dir / "holiday_demand.png")}

In [29]:
start = time.perf_counter()
prediction_review = test_data[["Fecha", "Origen_Id", TARGET_COLUMN]].copy()
prediction_review["predicted_trips"] = final_predictions
prediction_review["naive_prediction"] = naive_predictions
prediction_review["absolute_error"] = (
    prediction_review[TARGET_COLUMN] - prediction_review["predicted_trips"]
).abs()
prediction_review["segment"] = test_segments

figure_paths = {
    "mae_by_fold": plot_mae_by_fold(
        pd.concat([all_results, final_walk_forward], ignore_index=True),
        [NAIVE_BASELINE, "Promedio estación-día"]
        + [f"{name} | {MAIN_FEATURE_SET}" for name in MODEL_NAMES] + [final_label],
        MODEL_FIGURES_DIR / "mae_by_fold.png",
    ),
    "test_predictions": plot_test_predictions(
        prediction_review, TARGET_COLUMN, MODEL_FIGURES_DIR / "test_predictions_2026.png"
    ),
}
figure_paths.update(plot_exploratory_figures(integrated_data, POI_COLUMNS, MODEL_FIGURES_DIR / "exploration"))
figure_paths.update(plot_additional_eda(integrated_data, POI_COLUMNS, MODEL_FIGURES_DIR / "exploration"))
figure_paths.update(plot_holiday_demand(integrated_data, MODEL_FIGURES_DIR / "exploration"))
for path in figure_paths.values():
    print(f"Gráfica guardada sin abrir ventana: {path}")
stage_times["12_figures"] = time.perf_counter() - start

Gráfica guardada sin abrir ventana: /home/josel/mlops/equipo-8-mibici-mlops/reports/figures/modeling/mae_by_fold.png
Gráfica guardada sin abrir ventana: /home/josel/mlops/equipo-8-mibici-mlops/reports/figures/modeling/test_predictions_2026.png
Gráfica guardada sin abrir ventana: /home/josel/mlops/equipo-8-mibici-mlops/reports/figures/modeling/exploration/demand_distribution.png
Gráfica guardada sin abrir ventana: /home/josel/mlops/equipo-8-mibici-mlops/reports/figures/modeling/exploration/weekday_demand.png
Gráfica guardada sin abrir ventana: /home/josel/mlops/equipo-8-mibici-mlops/reports/figures/modeling/exploration/monthly_demand.png
Gráfica guardada sin abrir ventana: /home/josel/mlops/equipo-8-mibici-mlops/reports/figures/modeling/exploration/station_ranking.png
Gráfica guardada sin abrir ventana: /home/josel/mlops/equipo-8-mibici-mlops/reports/figures/modeling/exploration/station_poi.png
Gráfica guardada sin abrir ventana: /home/josel/mlops/equipo-8-mibici-mlops/reports/figures/m

## 11. Registro de experimentos en MLflow (`tracking.py`)

MLflow conserva las métricas y los archivos de cada etapa. Para que sea fácil encontrar cada resultado, organizamos **Artifacts** por datos, exploración, modelado, entorno y tiempos. Cada ejecución padre guarda la procedencia del dataset y su inventario SHA-256; las ejecuciones hijas muestran las métricas de cada mes y el hash del dataset usado.

Los archivos CSV originales y los Parquet grandes **no se duplican**. Se registra su ruta y versión para poder identificarlos. Los datasets de desarrollo y prueba se vinculan también a la ejecución final mediante `mlflow.log_input()`.

Las gráficas PNG se consultan en **Artifacts**. Las métricas numéricas están en **Model metrics**; **System metrics** se reserva para CPU y memoria cuando está habilitado.


In [30]:
def setup_mlflow(tracking_uri: str, experiment_name: str, artifact_dir: Path) -> str:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(tracking_uri)
    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        return mlflow.create_experiment(experiment_name, artifact_location=artifact_dir.as_uri())
    return experiment.experiment_id


def log_run_context(project_root: Path, data_path: Path, data_sha256: str) -> None:
    # Información compartida por todas las ejecuciones.
    environment = collect_environment(project_root)
    for package in ("scikit-learn", "scipy", "joblib"):
        environment["packages"][package] = metadata.version(package)
    mlflow.set_tags({
        "implementation_commit": environment["git_commit"] or "unavailable",
        "python_version": environment["python_version"],
        "sklearn_version": environment["packages"]["scikit-learn"],
        "data_sha256": data_sha256,
        "data_manifest": "data/lineage/data_provenance.json",
    })
    mlflow.log_dict(environment, "environment/runtime.json")
    # Configuración reproducible y documentación de parámetros comunes.
    run_parameters = {
        "data_file": data_path.name,
        "horizon_days": HORIZON_DAYS,
        "lags": LAGS,
        "rolling_windows": ROLLING_WINDOWS,
        "first_cutoff": FIRST_CUTOFF.date(),
        "test_start": TEST_START.date(),
        "n_folds": N_FOLDS,
        "seed": SEED,
    }
    mlflow.log_dict(
        {
            "parameters": {name: str(value) for name, value in run_parameters.items()},
            "data_sha256": data_sha256,
            "data_path": str(data_path),
            "environment": environment,
        },
        "documentation/run_configuration.json",
    )
    mlflow.log_params({
        "data_file": data_path.name,
        "horizon_days": HORIZON_DAYS,
        "lags": LAGS,
        "rolling_windows": ROLLING_WINDOWS,
        "first_cutoff": FIRST_CUTOFF.date(),
        "test_start": TEST_START.date(),
        "n_folds": N_FOLDS,
        "seed": SEED,
    })


def log_data_documentation(context: dict) -> None:
    """Guardar el inventario y la procedencia en la ejecución padre."""
    mlflow.log_artifact(str(CSV_VERSION_DIR / "csv_versions.csv"), "data/inventory")
    mlflow.log_artifact(str(CSV_VERSION_DIR / "csv_versions.json"), "data/inventory")
    mlflow.log_artifact(str(provenance_path), "data/lineage")
    mlflow.log_artifact(str(FILE_VERSION_TABLE), "data/versions")
    mlflow.log_artifact(str(FILE_VERSION_INDEX), "data/versions")
    mlflow.log_dict({
        "integrated_path": str(context["data_path"].relative_to(PROJECT_ROOT)),
        "integrated_sha256": context["data_sha256"],
        "phase1_run": data_provenance["phase1_run"],
        "csv_inventory_count": len(csv_inventory),
        "file_inventory_count": int(file_inventory["file"].nunique()),
        "file_versions_manifest": "data/versions/file_versions.csv",
    }, "data/lineage/dataset_reference.json")


def log_dataset_reference(context: dict, role: str = "validation") -> None:
    """Vincular cada corrida con el dataset integrado, sin copiar sus filas."""
    mlflow.set_tags({
        "dataset_role": role,
        "dataset_path": str(context["data_path"].relative_to(PROJECT_ROOT)),
        "dataset_sha256": context["data_sha256"],
    })
    log_data_documentation(context)
    # Mismo dataset de desarrollo para los modelos y las líneas base.
    mlflow.log_input(MLFLOW_DEVELOPMENT_INPUT, context="training" if role == "training_and_testing" else "validation")
    if role == "training_and_testing":
        mlflow.log_input(MLFLOW_TEST_INPUT, context="testing")


def log_report_artifacts(files: list[Path]) -> list[dict]:
    """Separar los archivos por contenido, no por la carpeta donde se generaron."""
    eda_demand = {
        "demand_distribution.png", "monthly_demand.png", "monthly_demand_trend.png",
        "weekday_demand.png", "day_of_month_demand.png", "holiday_demand.png",
        "missing_periods.png", "shared_missing_dates.png", "demand_correlation.png",
    }
    eda_stations = {
        "station_ranking.png", "annual_station_coverage.png",
        "station_attribute_consistency.png",
    }
    eda_external = {
        "weather_distribution.png", "weather_vs_demand.png", "station_poi.png",
        "top_station_demand_vs_poi.png", "bottom_station_demand_vs_poi.png",
    }
    modeling_validation = {"mae_by_fold.png", "final_walk_forward.csv"}
    modeling_comparison = {"model_comparison.csv", "mlflow_model_comparison.png"}
    modeling_test = {"test_results_2026.csv", "test_predictions_2026.png"}
    quality = {"quality_report.csv"}
    inventory = {"csv_versions.csv", "csv_versions.json"}
    recorded = set()
    catalog = []
    for path in files:
        path = Path(path)
        if not path.is_file():
            raise FileNotFoundError(f"Artefacto no encontrado: {path}")
        if path.name in inventory:
            folder = "data/inventory/model_reports" if "model_reports" in path.parts else "data/inventory"
        elif path.name in quality:
            folder = "data/quality"
        elif path.name in eda_demand:
            folder = "eda/fase_1/demanda"
        elif path.name in eda_stations:
            folder = "eda/fase_1/estaciones"
        elif path.name in eda_external:
            folder = "eda/fase_1/fuentes_externas"
        elif path.name in modeling_validation:
            folder = "modeling/validation"
        elif path.name in modeling_comparison:
            folder = "modeling/comparison"
        elif path.name in modeling_test:
            folder = "modeling/test"
        else:
            folder = "reports/other"
        # Los análisis complementarios del notebook 05 van en Fase 2.
        phase2_eda = {
            "monthly_demand_trend.png": "demanda",
            "day_of_month_demand.png": "demanda",
            "holiday_demand.png": "demanda",
            "missing_periods.png": "calidad_temporal",
            "shared_missing_dates.png": "calidad_temporal",
            "annual_station_coverage.png": "estaciones",
            "station_attribute_consistency.png": "estaciones",
            "top_station_demand_vs_poi.png": "fuentes_externas",
            "bottom_station_demand_vs_poi.png": "fuentes_externas",
        }
        if path.name in phase2_eda:
            folder = f"eda/fase_2/{phase2_eda[path.name]}"
        # Evitar duplicados en una misma carpeta, sin descartar ningún archivo.
        key = (folder, path.name)
        if key not in recorded:
            mlflow.log_artifact(str(path), folder)
            recorded.add(key)
            catalog.append({
                "file": path.name,
                "source_path": path.relative_to(PROJECT_ROOT).as_posix(),
                "artifact_path": f"{folder}/{path.name}",
                "size_bytes": path.stat().st_size,
                "sha256": calculate_sha256(path),
            })
    return catalog


MLFLOW_ARTIFACT_GUIDE = """# Guía de artefactos de MiBici

- `data/inventory`: listado de CSV encontrados y sus versiones SHA-256.
- `data/inventory/model_reports`: versiones de los CSV de resultados.
- `data/lineage`: procedencia y referencias de datasets.
- `data/versions`: SHA-256 y versiones de CSV, Parquet, PBF y demás archivos de datos.
- `data/outputs`: hashes y versiones de los archivos de resultados de esta ejecución.
- `data/quality`: comprobaciones de calidad del dataset.
- `eda/fase_1`: gráficas exploratorias recuperadas del análisis inicial (demanda, estaciones y fuentes externas).
- `eda/fase_2`: gráficas complementarias del notebook de modelado (demanda, calidad temporal, estaciones y POI).
- Esta separación indica la etapa del análisis; no implica que sean copias exactas de los PNG originales de Fase 1.
- `modeling/validation`: resultados y gráficas de validación temporal.
- `modeling/comparison`: comparación de modelos y líneas base.
- `modeling/test`: resultados de la prueba independiente de 2026.
- `model`: pipeline serializado y firma de entrada de MLflow.
- `environment`: entorno, versiones de librerías y commit.
- `performance`: tiempos por etapa (`execution_times.csv`), gráfica comparativa (`execution_times.png`), resumen y versiones SHA-256.
- `documentation`: configuración y parámetros de cada ejecución y del modelo final.

Los CSV originales y los Parquet grandes no se copian a MLflow. El manifiesto registra
su ruta y hash cuando están disponibles. Los CSV del inventario no son necesariamente
todos los que se utilizaron para entrenar.

Las gráficas PNG se consultan en Artifacts. Los valores de MAE y RMSE por mes
se registran también como series de métricas para consultarlos en Model metrics.
Las gráficas de demanda por día, mes, evolución mensual y cobertura anual se
registran también como series en Model metrics. Sus pasos se explican en
`eda/metric_series_guide.json`. Histogramas, POI y relaciones con clima se
conservan como PNG en Artifacts, porque no tienen una escala temporal común.
System metrics corresponde al consumo de recursos, si está habilitado.
"""


def log_walk_forward(results, run_name, context, params=None, columns=None, tags=None) -> str:
    # Corrida principal con el resumen y una corrida hija por ventana.
    with mlflow.start_run(run_name=run_name) as run:
        log_run_context(**context)
        mlflow.set_tags({"stage": "walk_forward", **(tags or {})})
        log_dataset_reference(context, role="walk_forward")
        if params:
            mlflow.log_params({f"model_{k}": v for k, v in params.items()})
        if columns:
            mlflow.log_param("n_features", len(columns))
            mlflow.log_dict({"features": list(columns)}, "features.json")
        summary = {
            "mae_mean": results["mae"].mean(),
            "mae_std": results["mae"].std(),
            "rmse_mean": results["rmse"].mean(),
            "skill_vs_naive_mean": results["skill_vs_naive"].mean(),
        }
        if "fit_seconds" in results:
            summary["fit_seconds_total"] = results["fit_seconds"].sum()
        mlflow.log_metrics({k: float(v) for k, v in summary.items() if pd.notna(v) and np.isfinite(v)})
        for step, row in enumerate(results.itertuples(index=False)):
            mlflow.log_metric("fold_mae", row.mae, step=step)
            mlflow.log_metric("fold_rmse", row.rmse, step=step)
            with mlflow.start_run(run_name=f"{run_name} | {row.cutoff}", nested=True):
                mlflow.log_param("cutoff", row.cutoff)
                mlflow.set_tag("dataset_sha256", context["data_sha256"])
                fold_metrics = {"mae": row.mae, "rmse": row.rmse, "skill_vs_naive": row.skill_vs_naive}
                for name in ("fit_seconds", "train_rows", "validation_rows"):
                    if hasattr(row, name):
                        fold_metrics[name] = getattr(row, name)
                mlflow.log_metrics({k: float(v) for k, v in fold_metrics.items() if pd.notna(v) and np.isfinite(v)})
        mlflow.log_table(results.astype({"cutoff": str}), "modeling/validation/walk_forward_results.json")
        # Una gráfica por modelo, guardada en la corrida padre y sin ventanas.
        if not results.empty:
            fig, ax = plt.subplots(figsize=(9, 4))
            ax.plot(results["cutoff"].astype(str), results["mae"], marker="o", label="MAE")
            ax.plot(results["cutoff"].astype(str), results["rmse"], marker="s", label="RMSE")
            ax.set(title="Error por ventana temporal", xlabel="Inicio de validación", ylabel="Viajes")
            ax.tick_params(axis="x", rotation=60)
            ax.legend()
            fig.tight_layout()
            chart_path = MODEL_FIGURES_DIR / f"mlflow_{run.info.run_id}_fold_errors.png"
            save_figure(fig, chart_path)
            mlflow.log_artifact(str(chart_path), artifact_path="modeling/validation")
        return run.info.run_id


def log_grid_search(search, model_name: str, context: dict) -> str:
    with mlflow.start_run(run_name=f"Búsqueda | {model_name}") as run:
        log_run_context(**context)
        mlflow.set_tag("stage", "tuning")
        log_dataset_reference(context, role="tuning")
        results = pd.DataFrame(search.cv_results_)
        for i, row in results.iterrows():
            with mlflow.start_run(run_name=f"{model_name} | combinación {i}", nested=True):
                mlflow.log_params({k.replace("model__", ""): v for k, v in row["params"].items()})
                mlflow.log_metrics({
                    "mae": -row["mean_test_mae"],
                    "mae_std": row["std_test_mae"],
                    "rmse": -row["mean_test_rmse"],
                    "fit_seconds": row["mean_fit_time"],
                })
        return run.info.run_id


def log_eda_metric_series(data: pd.DataFrame, poi_columns: list[str]) -> None:
    """Guardar indicadores de EDA que sí tienen sentido como métricas numéricas."""
    guide = {}

    def log_series(name: str, values: pd.Series, meaning: str, unit: str) -> None:
        # MLflow usa pasos numéricos; las etiquetas originales quedan en el catálogo.
        series = pd.Series(values)
        for step, value in enumerate(series):
            if pd.notna(value) and np.isfinite(value):
                mlflow.log_metric(name, float(value), step=step)
        guide[name] = {
            "description": meaning,
            "unit": unit,
            "source": "Dataset integrado utilizado en esta ejecución",
            "interpretation": "Cada punto corresponde a la etiqueta de la misma posición; no es un instante de entrenamiento.",
            "labels": [str(label) for label in series.index],
            "step": "Posición de la etiqueta en labels, empezando en 0",
        }

    def log_value(name: str, value: float, meaning: str, unit: str) -> None:
        if pd.notna(value) and np.isfinite(value):
            mlflow.log_metric(name, float(value))
            guide[name] = {"description": meaning, "unit": unit, "source": "Dataset integrado utilizado en esta ejecución", "step": "Métrica escalar: no tiene serie temporal"}

    # Definiciones explícitas: evitar descripciones como "Resumen: p90_trips".
    summary_definitions = {
        "records": ("Registros estación-día disponibles en el dataset integrado", "registros"),
        "stations": ("Estaciones de origen distintas en el dataset integrado", "estaciones"),
        "dates": ("Fechas distintas con registros en el dataset integrado", "días"),
        "total_trips": ("Suma de viajes de todos los registros estación-día", "viajes"),
        "mean_trips": ("Promedio de viajes por registro estación-día", "viajes/estación-día"),
        "median_trips": ("Mediana de viajes por registro estación-día", "viajes/estación-día"),
        "std_trips": ("Desviación estándar muestral de viajes por estación-día", "viajes/estación-día"),
        "min_trips": ("Menor número de viajes registrado en una estación-día", "viajes"),
        "max_trips": ("Mayor número de viajes registrado en una estación-día", "viajes"),
        "p10_trips": ("Percentil 10 de viajes por estación-día", "viajes/estación-día"),
        "p90_trips": ("Percentil 90 de viajes por estación-día", "viajes/estación-día"),
        "p95_trips": ("Percentil 95 de viajes por estación-día", "viajes/estación-día"),
        "zero_trip_rows": ("Registros estación-día con cero viajes", "registros"),
        "zero_trip_fraction": ("Proporción de registros estación-día con cero viajes", "proporción 0–1"),
    }
    target = data["target"]
    summary_values = {
        "records": len(data),
        "stations": data["Origen_Id"].nunique(),
        "dates": data["Fecha"].nunique(),
        "total_trips": target.sum(),
        "mean_trips": target.mean(),
        "median_trips": target.median(),
        "std_trips": target.std(),
        "min_trips": target.min(),
        "max_trips": target.max(),
        "p10_trips": target.quantile(0.10),
        "p90_trips": target.quantile(0.90),
        "p95_trips": target.quantile(0.95),
        "zero_trip_rows": target.eq(0).sum(),
        "zero_trip_fraction": target.eq(0).mean(),
    }
    for name, value in summary_values.items():
        description, unit = summary_definitions[name]
        log_value(f"eda_fase1_{name}", value, description, unit)

    log_series(
        "eda_fase1_weekday_mean_trips",
        data.groupby("day_of_week")["target"].mean().reindex(range(7)),
        "Promedio de viajes por día de la semana; 0=lunes", "viajes/estación-día",
    )
    log_series(
        "eda_fase1_month_mean_trips",
        data.groupby("month")["target"].mean().reindex(range(1, 13)),
        "Promedio de viajes por mes del año", "viajes/estación-día",
    )
    station_mean = data.groupby("Origen_Id")["target"].mean()
    for name, value in {
        "min": station_mean.min(),
        "p10": station_mean.quantile(0.10),
        "median": station_mean.median(),
        "p90": station_mean.quantile(0.90),
        "max": station_mean.max(),
    }.items():
        log_value(f"eda_fase1_station_mean_{name}", value,
                  f"Estadístico {name} de los promedios diarios de viajes calculados por estación", "viajes/estación-día")

    # Fase 2: tendencia y cobertura; una observación por mes o por año.
    monthly_group = data.groupby(data["Fecha"].dt.to_period("M"))
    monthly = monthly_group["target"].agg(["mean", "sum", "size"])
    monthly["stations"] = monthly_group["Origen_Id"].nunique()
    for column, meaning, unit in (
        ("mean", "Demanda media por estación-día observado", "viajes/estación-día"),
        ("sum", "Total mensual de viajes", "viajes"),
        ("size", "Registros estación-día observados", "registros"),
        ("stations", "Estaciones con registros en el mes", "estaciones"),
    ):
        log_series(f"eda_fase2_monthly_{column}", monthly[column], meaning, unit)

    annual_group = data.groupby(data["Fecha"].dt.year)
    annual = annual_group["target"].agg(["sum", "mean", "size"])
    annual["stations"] = annual_group["Origen_Id"].nunique()
    for column, unit in (("sum", "viajes"), ("mean", "viajes/estación-día"),
                         ("size", "registros"), ("stations", "estaciones")):
        log_series(f"eda_fase2_annual_{column}", annual[column],
                   f"Estadístico anual {column} de demanda o cobertura de estaciones", unit)

    log_series(
        "eda_fase2_day_of_month_mean_trips",
        data.groupby(data["Fecha"].dt.day)["target"].mean().reindex(range(1, 32)),
        "Demanda media por día del mes", "viajes/estación-día",
    )

    # Cobertura: faltantes dentro de la vida observada de cada estación.
    spans = data.groupby("Origen_Id")["Fecha"].agg(["min", "max"])
    all_dates = pd.date_range(data["Fecha"].min(), data["Fecha"].max(), freq="D")
    starts = spans["min"].value_counts().reindex(all_dates, fill_value=0).cumsum()
    ends = spans["max"].value_counts().reindex(all_dates, fill_value=0).cumsum().shift(1, fill_value=0)
    expected = starts.sub(ends)
    observed = data.groupby("Fecha")["Origen_Id"].nunique().reindex(all_dates, fill_value=0)
    missing = expected.sub(observed).clip(lower=0)
    monthly_missing = missing.groupby(missing.index.to_period("M")).sum()
    log_series("eda_fase2_monthly_missing_station_days", monthly_missing,
               "Suma mensual de estación-días sin registro dentro de su periodo observado",
               "estación-días")
    log_value("eda_fase2_missing_station_days_total", missing.sum(),
              "Total de estación-días sin registro", "estación-días")
    log_value("eda_fase2_missing_station_days_fraction", missing.sum() / expected.sum()
              if expected.sum() else 0,
              "Fracción de estación-días sin registro", "proporción")

    previous = data[["Origen_Id", "Fecha"]].sort_values(
        ["Origen_Id", "Fecha"]
    ).groupby("Origen_Id")["Fecha"].shift()
    gap_days = (data["Fecha"] - previous).dt.days.sub(1)
    # El desplazamiento anterior conserva los índices originales.
    gap_days = gap_days.loc[gap_days > 0]
    log_value("eda_fase2_missing_periods_count", len(gap_days),
              "Cantidad de intervalos sin registros", "intervalos")
    log_value("eda_fase2_missing_periods_max_days", gap_days.max() if len(gap_days) else 0,
              "Duración máxima de un intervalo sin registros", "días")

    # Calidad: únicamente indicadores calculables con el dataset integrado.
    log_value("quality_duplicate_station_dates",
              data.duplicated(["Origen_Id", "Fecha"]).sum(),
              "Filas con estación y fecha repetidas", "filas")
    log_value("quality_target_nulls", target.isna().sum(),
              "Valores nulos en la demanda", "filas")
    log_value("quality_negative_trips", target.lt(0).sum(),
              "Filas con demanda negativa", "filas")

    # POI: resumen de disponibilidad y consistencia, no barras por categoría.
    present_poi = [column for column in poi_columns if column in data.columns]
    if present_poi:
        station_poi = data.drop_duplicates("Origen_Id").set_index("Origen_Id")[present_poi]
        log_value("eda_fase2_poi_columns", len(present_poi),
                  "Número de columnas de POI", "columnas")
        log_value("eda_fase2_stations_with_any_poi",
                  station_poi.fillna(0).gt(0).any(axis=1).sum(),
                  "Estaciones con al menos un POI registrado", "estaciones")
        changes = data.groupby("Origen_Id")[present_poi].nunique(dropna=False).gt(1)
        log_value("quality_stations_with_changing_poi", changes.any(axis=1).sum(),
                  "Estaciones cuyos atributos POI cambian en el tiempo", "estaciones")
    if "weather_zone" in data.columns:
        log_value("eda_fase2_weather_zones", data["weather_zone"].nunique(),
                  "Zonas meteorológicas presentes", "zonas")

    # Temperatura y lluvia son exploratorias; nunca se agregan a las 18 variables.
    for column, unit in (("temperature_2m_mean", "°C"),
                         ("precipitation_sum", "mm")):
        if column in data.columns:
            values = data[column]
            log_value(f"eda_fase1_{column}_missing_fraction", values.isna().mean(),
                      f"Fracción de datos faltantes de {column}", "proporción")
            log_value(f"eda_fase1_{column}_mean", values.mean(),
                      f"Media exploratoria de {column}", unit)

    # Las etiquetas y las unidades permiten interpretar cada serie en MLflow.
    mlflow.log_dict(guide, "eda/metric_series_guide.json")
    # Guía legible sin abrir JSON; MLflow no admite descripciones nativas por métrica.
    lines = [
        "# Diccionario de métricas exploratorias",
        "",
        "Todas estas métricas se calculan sobre el dataset integrado de esta ejecución.",
        "Una fila representa una estación en una fecha; `target` es su número de viajes.",
        "Fase 1 y Fase 2 indican el origen del análisis, no datasets independientes.",
        "Las proporciones se expresan entre 0 y 1; multiplícalas por 100 para obtener %.",
        "Las series utilizan `step` como índice; consulta las etiquetas para interpretar fechas y categorías.",
        "Las métricas describen los datos disponibles, no necesariamente la población total de viajes.",
        "Las visualizaciones originales permanecen como PNG en Artifacts.",
        "",
        "| Métrica | Qué mide | Unidad |",
        "|---|---|---|",
    ]
    for name, info in sorted(guide.items()):
        lines.append(
            f"| `{name}` | {info['description']} | {info['unit']} |"
        )
    lines.extend([
        "",
        "## Series y etiquetas",
        "En `metric_series_guide.json`, `labels` asigna a cada step su mes, año o día.",
        "No compares valores con unidades distintas en el mismo eje.",
    ])
    mlflow.log_text("\\n".join(lines), "documentation/EDA_METRICS_README.md")


def log_final_model(pipeline, model_name, params, validation, test, example, context, artifacts):
    # Corrida del modelo final, con el modelo registrado en el Model Registry.
    with mlflow.start_run(run_name=f"Modelo final | {model_name}") as run:
        log_run_context(**context)
        mlflow.set_tags({"stage": "final_model", "model_family": model_name})
        log_dataset_reference(context, role="training_and_testing")
        mlflow.log_text(MLFLOW_ARTIFACT_GUIDE, "README.md")
        mlflow.log_params({f"model_{k}": v for k, v in params.items()})
        mlflow.log_param("feature_set", MAIN_FEATURE_SET)
        mlflow.log_param("n_features", len(example.columns))
        mlflow.log_param("registered_model_name", REGISTERED_MODEL_NAME)
        mlflow.log_param("registered_model_alias", MODEL_ALIAS)
        mlflow.log_dict(
            {
                "model_family": model_name,
                "model_parameters": {key: str(value) for key, value in params.items()},
                "feature_set": MAIN_FEATURE_SET,
                "features": list(example.columns),
                "target": TARGET_COLUMN,
                "forecast_horizon_days": HORIZON_DAYS,
                "validation_folds": N_FOLDS,
                "test_start": str(TEST_START.date()),
                "random_seed": SEED,
            },
            "documentation/model_configuration.json",
        )
        mlflow.log_dict({"features": list(example.columns)}, "features.json")
        # Los datasets ya se vincularon mediante log_dataset_reference.
        final_metrics = {
            "validation_mae_mean": validation["mae"].mean(),
            "validation_mae_std": validation["mae"].std(),
            "validation_rmse_mean": validation["rmse"].mean(),
            "validation_skill_vs_naive": validation["skill_vs_naive"].mean(),
            "final_fit_seconds": float(artifacts.get("fit_seconds", 0.0)),
        }
        mlflow.log_metrics({k: float(v) for k, v in final_metrics.items() if pd.notna(v) and np.isfinite(v)})
        for row in test.itertuples(index=False):
            prefix = {"Todas": "test", "Con historia": "test_known", "Nueva": "test_new"}[row.stations]
            segment_metrics = {
                f"{prefix}_mae": row.model_mae,
                f"{prefix}_rmse": row.model_rmse,
                f"{prefix}_naive_mae": row.naive_mae,
                f"{prefix}_skill_vs_naive": row.skill_vs_naive,
                f"{prefix}_rows": row.rows,
            }
            mlflow.log_metrics({k: float(v) for k, v in segment_metrics.items() if pd.notna(v) and np.isfinite(v)})
        # Estas series se pueden graficar directamente en Model metrics.
        for step, row in enumerate(validation.itertuples(index=False)):
            mlflow.log_metric("validation_fold_mae", float(row.mae), step=step)
            mlflow.log_metric("validation_fold_rmse", float(row.rmse), step=step)
        log_eda_metric_series(artifacts["eda_data"], POI_COLUMNS)
        artifact_catalog = log_report_artifacts(artifacts["files"])
        mlflow.log_dict({"files": artifact_catalog}, "artifact_catalog.json")
        mlflow.log_artifact(str(FILE_VERSION_TABLE), "data/versions")
        mlflow.log_artifact(str(FILE_VERSION_INDEX), "data/versions")
        if "output_version_table" in artifacts:
            mlflow.log_artifact(str(artifacts["output_version_table"]), "data/outputs")
            mlflow.log_artifact(str(artifacts["output_version_index"]), "data/outputs")
        signature = infer_signature(example, pipeline.predict(example))
        model_info = mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="model",
            serialization_format="cloudpickle",
            signature=signature,
            input_example=example,
            registered_model_name=REGISTERED_MODEL_NAME,
        )
        return run.info.run_id, model_info


def promote_model(model_name: str, version: str, alias: str, description: str, tags: dict) -> None:
    client = mlflow.MlflowClient()
    client.update_model_version(model_name, version, description=description)
    for key, value in tags.items():
        client.set_model_version_tag(model_name, version, key, str(value))
    client.set_registered_model_alias(model_name, alias, version)

### 11.1 Registro de las corridas
Registramos todo lo que calculamos en las secciones 8 y 9. Las corridas se registran después de evaluar, con los resultados ya guardados en tablas; así, si MLflow falla, no se pierde el entrenamiento.

1. Una corrida por línea base y una por modelo con parámetros iniciales (y, si se ejecutaron, la comparación de conjuntos de variables y el clima).
2. Si hubo búsqueda, una corrida por modelo con una hija por combinación.
3. La validación del modelo final en las 12 ventanas.
4. El modelo final con la prueba de 2026, registrado y promovido a `champion`.

In [31]:
start = time.perf_counter()
experiment_id = setup_mlflow(MLFLOW_TRACKING_URI, MODELING_EXPERIMENT, MLFLOW_ARTIFACT_DIR)
mlflow.set_experiment(experiment_id=experiment_id)
data_sha256 = calculate_sha256(INTEGRATED_PATH)
run_context = {"project_root": PROJECT_ROOT, "data_path": INTEGRATED_PATH, "data_sha256": data_sha256}
# Construimos las referencias una sola vez y las reutilizamos en todas las corridas.
MLFLOW_DEVELOPMENT_INPUT = mlflow.data.from_pandas(
    development_data[[TARGET_COLUMN] + FEATURE_COLUMNS],
    source=str(INTEGRATED_PATH),
    name="mibici_development_2020_2025",
    targets=TARGET_COLUMN,
)
MLFLOW_TEST_INPUT = mlflow.data.from_pandas(
    test_data[[TARGET_COLUMN] + FEATURE_COLUMNS],
    source=str(INTEGRATED_PATH),
    name="mibici_test_2026",
    targets=TARGET_COLUMN,
)

for baseline_name in BASELINES:
    log_walk_forward(
        baseline_results.loc[baseline_results["model"] == baseline_name],
        f"Línea base | {baseline_name}", run_context, tags={"method": "baseline"},
    )
for label, run in model_runs.items():
    log_walk_forward(
        run["results"], label, run_context, params=run["params"],
        columns=FEATURE_SETS[run["feature_set"]],
        tags={"method": "model", "feature_set": run["feature_set"], "params": "initial"},
    )
for model_name, search in grid_searches.items():
    log_grid_search(search, model_name, run_context)
log_walk_forward(
    final_walk_forward, f"Validación | {final_label}", run_context,
    params=final_params, columns=FEATURE_COLUMNS,
    tags={"method": "model", "feature_set": MAIN_FEATURE_SET, "params": "final"},
)
print(f"Experimento: {MODELING_EXPERIMENT} ({experiment_id})")
stage_times["13_mlflow_validation_runs"] = time.perf_counter() - start
# Comparación resumida para MLflow (los hijos conservan el detalle mensual).
comparison = model_comparison.reset_index()
if not comparison.empty and "mae" in comparison.columns:
    labels = comparison.iloc[:, 0].astype(str)
    fig, ax = plt.subplots(figsize=(11, max(4, len(comparison) * 0.35)))
    ax.barh(labels, comparison["mae"])
    ax.invert_yaxis()
    ax.set(xlabel="MAE promedio (viajes)", title="Comparación de modelos y líneas base")
    fig.tight_layout()
    comparison_path = save_figure(fig, MODEL_FIGURES_DIR / "mlflow_model_comparison.png")
else:
    comparison_path = None

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values a

Experimento: MiBici_Fase2_Modelado (3)


In [32]:
start = time.perf_counter()
MODEL_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
report_files = []
for name, table in {
    "model_comparison.csv": model_comparison.reset_index(),
    "final_walk_forward.csv": final_walk_forward,
    "test_results_2026.csv": test_results,
    "quality_report.csv": quality_report,
}.items():
    path = MODEL_REPORTS_DIR / name
    table.to_csv(path, index=False)
    report_files.append(path)

# Versionar también los CSV de resultados, conservando los hashes anteriores.
start_versioning = time.perf_counter()
report_inventory = version_csv_sources(MODEL_REPORTS_DIR, CSV_VERSION_DIR / "model_reports")
report_files.extend([
    CSV_VERSION_DIR / "model_reports" / "csv_versions.json",
    CSV_VERSION_DIR / "model_reports" / "csv_versions.csv",
])
stage_times["14_csv_report_versioning"] = time.perf_counter() - start_versioning

# Versionar todos los resultados usados en esta ejecución, incluidos los PNG.
output_files = report_files + list(figure_paths.values())
if comparison_path is not None:
    output_files.append(comparison_path)
output_inventory = version_used_files(
    [(path, "resultado_generado", "modelado") for path in output_files]
)
# Copias del índice actual para no sobrescribir el inventario de entradas en MLflow.
OUTPUT_VERSION_DIR = CSV_VERSION_DIR / "model_outputs"
OUTPUT_VERSION_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VERSION_TABLE = OUTPUT_VERSION_DIR / "file_versions.csv"
OUTPUT_VERSION_INDEX = OUTPUT_VERSION_DIR / "file_versions.json"
output_inventory.loc[
    output_inventory["stage"] == "modelado"
].to_csv(OUTPUT_VERSION_TABLE, index=False)
shutil.copy2(FILE_VERSION_INDEX, OUTPUT_VERSION_INDEX)

final_run_id, model_info = log_final_model(
    final_pipeline, final_model_name, final_params,
    final_walk_forward, test_results,
    example=development_data[FEATURE_COLUMNS].head(5),
    context=run_context,
    artifacts={"files": report_files + list(figure_paths.values()) + ([comparison_path] if comparison_path else []), "fit_seconds": final_fit_seconds,
                "eda_data": integrated_data,
                "output_version_table": OUTPUT_VERSION_TABLE,
                "output_version_index": OUTPUT_VERSION_INDEX,
               },
)
model_version = str(model_info.registered_model_version)
test_all = test_results.set_index("stations").loc["Todas"]
promote_model(
    REGISTERED_MODEL_NAME, model_version, MODEL_ALIAS,
    description=(
        f"{final_model_name} para predecir las salidas diarias por estación con horizonte de "
        f"{HORIZON_DAYS} día. Variables: {MAIN_FEATURE_SET} ({len(FEATURE_COLUMNS)}), sin clima. "
        f"Entrenado con {development_data['Fecha'].min().date()} a "
        f"{development_data['Fecha'].max().date()}."
    ),
    tags={
        "validation_mae": round(final_walk_forward["mae"].mean(), 3),
        "test_mae": round(test_all["model_mae"], 3),
        "test_naive_mae": round(test_all["naive_mae"], 3),
        "feature_set": MAIN_FEATURE_SET,
        "data_sha256": data_sha256,
        "data_manifest": "data/lineage/data_provenance.json",
        "source_run_id": final_run_id,
    },
)
print(f"Corrida del modelo final: {final_run_id}")
print(f"Modelo registrado: {REGISTERED_MODEL_NAME}, versión {model_version}, alias '{MODEL_ALIAS}'")
stage_times["14_mlflow_registry"] = time.perf_counter() - start

/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missi

Corrida del modelo final: a9e63be2fc494bada8c32498c8ab118a
Modelo registrado: mibici_salidas_diarias, versión 3, alias 'champion'


## 12. Predicción con el modelo registrado (`modeling/predict.py`)
El `predict.py` actual es la plantilla de Cookiecutter. Lo reemplazamos con lo que haría el equipo de operación cada noche: cargar el modelo `champion` y predecir las salidas del día siguiente.

- `load_registered_model`: carga el modelo desde el Registry por su alias, así no hay que saber qué versión es la vigente.
- `build_next_day_rows`: crea una fila por estación para la fecha a predecir. Toma las estaciones con actividad en los últimos 28 días, copia sus atributos fijos (zona y POI) y agrega el calendario de esa fecha.
- `predict_next_day`: une esas filas al historial, calcula los rezagos con la **misma** `add_history_features` del entrenamiento (así no puede haber diferencias entre cómo se entrena y cómo se predice) y aplica el modelo.

Primero comprobamos que el modelo cargado del Registry da exactamente las mismas predicciones de 2026 que el modelo en memoria. Después predecimos el día siguiente al último dato.

In [33]:
def load_registered_model(model_name: str, alias: str):
    return mlflow.sklearn.load_model(f"models:/{model_name}@{alias}")


def build_next_day_rows(data: pd.DataFrame, target_date, active_days: int = 28) -> pd.DataFrame:
    # Una fila por estación activa con el calendario de la fecha a predecir.
    recent = data.loc[
        (data["Fecha"] < target_date)
        & (data["Fecha"] >= target_date - pd.Timedelta(days=active_days))
    ]
    if recent.empty:
        raise ValueError("No hay estaciones con historial reciente para predecir.")
    stations = recent.sort_values("Fecha").groupby("Origen_Id")[STATION_COLUMNS].last()
    rows = stations.reset_index()
    rows["Fecha"] = pd.Timestamp(target_date)
    rows["day_of_week"] = rows["Fecha"].dt.day_of_week
    rows["month"] = rows["Fecha"].dt.month
    return rows


def predict_next_day(model, data: pd.DataFrame, target_date) -> pd.DataFrame:
    # Predecir las salidas por estación para target_date con la historia disponible.
    target_date = pd.Timestamp(target_date)
    history = data.loc[data["Fecha"] < target_date, ["Origen_Id", "Fecha", TARGET_COLUMN]]
    new_rows = build_next_day_rows(data.loc[data["Fecha"] < target_date], target_date)
    combined = pd.concat([history, new_rows[["Origen_Id", "Fecha"]]], ignore_index=True)
    with_history = add_history_features(combined, HORIZON_DAYS, LAGS, ROLLING_WINDOWS, TARGET_COLUMN)
    features = new_rows.merge(
        with_history.loc[with_history["Fecha"] == target_date, ["Origen_Id"] + HISTORY_COLUMNS],
        on="Origen_Id",
    )
    features = add_calendar_features(features)
    if features["Origen_Id"].duplicated().any():
        raise ValueError("La predicción contiene estaciones duplicadas.")
    features["predicted_trips"] = model.predict(features[FEATURE_COLUMNS])
    return features[["Origen_Id", "Fecha", "predicted_trips"] + HISTORY_COLUMNS]

In [34]:
start = time.perf_counter()
champion = load_registered_model(REGISTERED_MODEL_NAME, MODEL_ALIAS)
registry_predictions = champion.predict(test_data[FEATURE_COLUMNS])
print("Predicciones del Registry iguales al modelo en memoria:",
      np.allclose(registry_predictions, final_predictions))

next_date = model_dataset["Fecha"].max() + pd.Timedelta(days=1)
next_day = predict_next_day(champion, integrated_data, next_date)
print(f"Predicción para {next_date.date()}: {len(next_day)} estaciones, "
      f"{next_day['predicted_trips'].sum():,.0f} salidas en total")
display(next_day.sort_values("predicted_trips", ascending=False).head(10).round(1))
stage_times["15_registry_prediction"] = time.perf_counter() - start

Predicciones del Registry iguales al modelo en memoria: True
Predicción para 2026-10-01: 468 estaciones, 19,997 salidas en total


/tmp/ipykernel_1292/1370800724.py:11: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(next_day.sort_values("predicted_trips", ascending=False).head(10).round(1))


,Origen_Id,Fecha,predicted_trips,lag_1,lag_7,lag_14,roll_mean_7,active_days_7,roll_mean_28,active_days_28
47,51,2026-10-01,327.7,442.0,368.0,413.0,407.7,7.0,370.5,28.0
31,35,2026-10-01,324.2,378.0,355.0,337.0,326.3,7.0,267.9,27.0
246,271,2026-10-01,312.9,370.0,302.0,287.0,287.7,7.0,242.6,28.0
8,11,2026-10-01,234.0,279.0,246.0,221.0,222.7,7.0,203.8,28.0
48,52,2026-10-01,210.3,219.0,209.0,226.0,192.7,7.0,174.5,28.0
46,50,2026-10-01,188.8,224.0,176.0,188.0,235.6,7.0,204.2,28.0
235,260,2026-10-01,177.5,206.0,163.0,196.0,171.6,7.0,148.1,28.0
50,54,2026-10-01,174.1,191.0,167.0,187.0,185.9,7.0,177.2,28.0
138,157,2026-10-01,163.5,190.0,162.0,146.0,155.9,7.0,132.5,28.0
30,34,2026-10-01,163.1,176.0,192.0,154.0,148.1,7.0,129.4,27.0


También comprobamos que `predict_next_day` construye las mismas variables que el entrenamiento: predecimos el último día de 2026 que ya tiene datos y lo comparamos con las predicciones de la prueba para esa fecha.

In [35]:
start = time.perf_counter()
last_date = test_data["Fecha"].max()
check_day = predict_next_day(champion, integrated_data, last_date)

reference = prediction_review.loc[
    prediction_review["Fecha"] == last_date,
    ["Origen_Id", "predicted_trips"]
]
comparison = reference.merge(
    check_day[["Origen_Id", "predicted_trips"]],
    on="Origen_Id",
    suffixes=("_test", "_predict")
)

# Comprobar que no falten estaciones de evaluación.
missing_stations = (
    set(reference["Origen_Id"]) - set(check_day["Origen_Id"])
)

coverage_ok = len(missing_stations) == 0

# Comparar las predicciones de las estaciones comunes.
predictions_match = np.allclose(
    comparison["predicted_trips_test"],
    comparison["predicted_trips_predict"]
)

print("Estaciones comparadas:", len(comparison))
print("Estaciones esperadas:", len(reference))
print("Estaciones adicionales:", len(check_day) - len(comparison))
print("Cobertura de estaciones:", coverage_ok)
print("Mismas predicciones:", predictions_match)

assert coverage_ok, "Faltan estaciones de evaluación."
assert predictions_match, "Las predicciones no coinciden."

stage_times["16_prediction_consistency"] = time.perf_counter() - start

Estaciones comparadas: 467
Estaciones esperadas: 467
Estaciones adicionales: 1
Cobertura de estaciones: True
Mismas predicciones: True


### 12.1 Perfil de tiempos de ejecución
Separamos los tiempos de preparación, generación de variables, validación, entrenamiento, predicción, gráficas y MLflow. La tabla se ordena de mayor a menor para identificar cuellos de botella. Se guarda en CSV y se registra como artefacto del modelo final. Los tiempos dependen del equipo y de si se reutilizaron archivos Parquet.

In [36]:
timing_report = (
    pd.DataFrame(
        [{"stage": name, "seconds": seconds} for name, seconds in stage_times.items()]
    )
    .sort_values("seconds", ascending=False)
    .reset_index(drop=True)
)
MODEL_REPORTS_DIR.mkdir(parents=True, exist_ok=True)
timing_path = MODEL_REPORTS_DIR / "execution_times.csv"
timing_report.to_csv(timing_path, index=False)
display(timing_report.round(2))

with mlflow.start_run(run_id=final_run_id):
    mlflow.log_artifact(str(timing_path), artifact_path="performance")
    timing_version = version_used_files([(timing_path, "tiempos_ejecucion", "rendimiento")])
    mlflow.log_dict(timing_version.to_dict(orient="records"), "performance/file_version.json")
    # Cada etapa se consulta como métrica y también en el CSV completo.
    for name, seconds in stage_times.items():
        mlflow.log_metric(f"time_{name}_seconds", float(seconds))
    mlflow.log_metric("time_total_recorded_seconds", float(sum(stage_times.values())))
    mlflow.log_metric("time_stages_count", len(stage_times))
    mlflow.log_dict(
        {
            "unit": "seconds",
            "description": "Tiempo medido por etapa; no incluye trabajo anterior al inicio de los cronómetros.",
            "total_recorded_seconds": float(sum(stage_times.values())),
            "stages": [
                {"name": name, "seconds": float(seconds)}
                for name, seconds in stage_times.items()
            ],
        },
        "performance/execution_summary.json",
    )
    mlflow.log_text(
        "# Tiempos de ejecución\\n\\n"
        "Cada `time_<etapa>_seconds` indica segundos medidos para esa etapa.\\n"
        "`time_total_recorded_seconds` suma únicamente las etapas instrumentadas; "
        "no es el tiempo total de la sesión de Jupyter.\\n"
        "`time_stages_count` es la cantidad de etapas con medición.\\n"
        "Consulta `execution_times.csv` para los nombres completos y "
        "`execution_times.png` para compararlos visualmente.\\n",
        "performance/README.md",
    )
    # La comparación entre etapas se entiende mejor como gráfica de barras PNG.
    fig, ax = plt.subplots(figsize=(11, max(5, len(timing_report) * 0.32)))
    ax.barh(timing_report["stage"], timing_report["seconds"])
    ax.set_xlabel("Tiempo de ejecución (segundos)")
    ax.set_title("Tiempo por etapa del pipeline")
    ax.invert_yaxis()
    fig.tight_layout()
    timing_chart_path = MODEL_REPORTS_DIR / "execution_times.png"
    fig.savefig(timing_chart_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    mlflow.log_artifact(str(timing_chart_path), artifact_path="performance")
    chart_version = version_used_files([(timing_chart_path, "grafica_tiempos", "rendimiento")])
    mlflow.log_dict(chart_version.to_dict(orient="records"), "performance/chart_version.json")
print(f"Tiempos guardados en: {timing_path}")

,stage,seconds
0,09_final_walk_forward,821.78
1,06_cv_Random Forest_calendario_estacion_historia,755.61
2,10_final_fit,83.76
3,06_cv_HistGradientBoosting_calendario_estacion...,67.17
4,14_mlflow_registry,20.78
5,13_mlflow_validation_runs,18.48
6,12_figures,3.88
7,04_baselines,1.57
8,15_registry_prediction,1.06
9,03_validation_splits,1.06


Tiempos guardados en: /home/josel/mlops/equipo-8-mibici-mlops/reports/modeling/execution_times.csv


## 13. Punto de entrada (`main.py`)
El `main.py` actual corre siempre la Fase 1 completa: procesa los 81 CSV, rellena horas con cero, separa y hace el EDA. Para la Fase 2 proponemos un `main` por etapas que reutiliza las funciones de este notebook:

| Etapa | Qué hace | ¿Se salta si ya está hecho? |
|---|---|---|
| `data` | `prepare_model_data` | Sí, si el Parquet integrado pasa el contrato |
| `train` | Líneas base, modelo final en 12 ventanas, entrenamiento final, prueba 2026 y registro en MLflow | No; los interruptores de `config.py` controlan las partes lentas |
| `predict` | Carga `champion` y predice el día siguiente | — |

La Fase 1 sigue disponible con `MiBiciPipeline`; `main` solo avisa si no encuentra una ejecución válida. En el archivo `.py` se agrega al final el bloque para ejecutarlo desde la terminal:

```python
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Pipeline de la Fase 2 de MiBici")
    parser.add_argument("--stages", nargs="+", default=["data", "train", "predict"])
    main(parser.parse_args().stages)
```

Con eso se ejecuta `python -m mibici_mlops.main --stages data train`. Aquí solo definimos la función; no la llamamos porque repetiría todo el entrenamiento.

In [37]:
def run_training(development, test) -> dict:
    # Modelo final: validación, entrenamiento, prueba y registro.
    splits = walk_forward_splits(development["Fecha"], FIRST_CUTOFF, N_FOLDS, HORIZON_DAYS)
    baselines = evaluate_baselines(development, splits, TARGET_COLUMN)
    baselines = add_skill(baselines, baselines)
    model_name, params = best_configuration(None)
    pipeline = build_model_pipeline(make_model(model_name, params), FEATURE_COLUMNS)
    validation = add_skill(
        evaluate_walk_forward(pipeline, development, splits, FEATURE_COLUMNS, model_name),
        baselines,
    )
    start = time.perf_counter()
    final = train_final_model(model_name, params, development, FEATURE_COLUMNS, TARGET_COLUMN)
    fit_seconds = time.perf_counter() - start
    save_model(final, FINAL_MODEL_PATH)

    segments = np.where(
        test["Origen_Id"].isin(set(development["Origen_Id"])), "Con historia", "Nueva"
    )
    test_metrics = evaluate_by_segment(
        test[TARGET_COLUMN].to_numpy(),
        final.predict(test[FEATURE_COLUMNS]),
        predict_seasonal_naive(development, test, TARGET_COLUMN).to_numpy(),
        segments,
    )
    mlflow.set_experiment(
        experiment_id=setup_mlflow(MLFLOW_TRACKING_URI, MODELING_EXPERIMENT, MLFLOW_ARTIFACT_DIR)
    )
    context = {
        "project_root": PROJECT_ROOT,
        "data_path": INTEGRATED_PATH,
        "data_sha256": calculate_sha256(INTEGRATED_PATH),
    }
    for name in BASELINES:
        log_walk_forward(baselines.loc[baselines["model"] == name], f"Línea base | {name}", context)
    log_walk_forward(validation, f"Validación | {model_name} | final", context, params, FEATURE_COLUMNS)
    run_id, info = log_final_model(
        final, model_name, params, validation, test_metrics,
        development[FEATURE_COLUMNS].head(5), context,
        {"files": [], "fit_seconds": fit_seconds},
    )
    promote_model(
        REGISTERED_MODEL_NAME, str(info.registered_model_version), MODEL_ALIAS,
        f"{model_name}, {MAIN_FEATURE_SET}, horizonte {HORIZON_DAYS} día.",
        {"validation_mae": round(validation["mae"].mean(), 3)},
    )
    return {"run_id": run_id, "validation": validation, "test": test_metrics}


def main(stages=("data", "train", "predict")) -> dict:
    allowed = {"data", "train", "predict"}
    if not set(stages).issubset(allowed):
        raise ValueError(f"Etapas desconocidas: {sorted(set(stages) - allowed)}")
    outputs = {}
    integrated, report = prepare_model_data(force=FORCE_REBUILD_DATA)
    outputs["quality_report"] = report
    if "train" in stages:
        dataset = build_model_dataset(integrated, HORIZON_DAYS, LAGS, ROLLING_WINDOWS, TARGET_COLUMN)
        development, test = split_by_date(dataset, TEST_START)
        outputs["training"] = run_training(development, test)
    if "predict" in stages:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        model = load_registered_model(REGISTERED_MODEL_NAME, MODEL_ALIAS)
        target_date = integrated["Fecha"].max() + pd.Timedelta(days=1)
        outputs["predictions"] = predict_next_day(model, integrated, target_date)
        logger.info("Predicción generada para %s", target_date.date())
    return outputs

Para comprobar el punto de entrada sin repetir el entrenamiento, ejecutamos la etapa de datos. La predicción con el modelo registrado ya se comprueba en la sección 12.

In [38]:
start = time.perf_counter()
main_outputs = main(stages=("data",))
print(f"Pruebas de calidad aprobadas: {int(main_outputs['quality_report']['passed'].sum())} de 9")
stage_times["17_main_data_smoke_test"] = time.perf_counter() - start

2026-11-04 17:10:05,437 INFO mibici_mlops: Ejecución de la Fase 1 utilizada: mibici_20261026T025817_983267Z
2026-11-04 17:10:05,927 INFO mibici_mlops: Dataset integrado reutilizado: /home/josel/mlops/equipo-8-mibici-mlops/data/processed/mibici_integrated.parquet


Pruebas de calidad aprobadas: 9 de 9


## 14. Pruebas de las funciones nuevas (`tests/`)
Las pruebas actuales de `tests/` son de la Fase 1. Estas cuatro cubren lo que más nos importa de la Fase 2: que no se use información futura y que las ventanas y líneas base hagan lo que dicen. Usan una tabla pequeña inventada, así que corren en menos de un segundo. En el repositorio irían en `tests/test_features.py` y `tests/test_evaluate.py`; aquí las llamamos directamente.

- `test_lags_use_calendar_days`: si falta un día, `lag_1` del día siguiente debe ser `NaN`, no el valor de dos días antes.
- `test_lag_shorter_than_horizon_fails`: con horizonte 3, pedir `lag_1` debe dar error.
- `test_walk_forward_has_no_overlap`: en cada ventana, todas las fechas de entrenamiento son anteriores al corte y no hay filas compartidas.
- `test_seasonal_naive_uses_last_week`: el ingenuo devuelve `lag_7` cuando existe y el promedio de la estación cuando falta.

In [39]:
def make_toy_data() -> pd.DataFrame:
    dates = pd.date_range("2024-12-01", "2025-02-28")
    data = pd.DataFrame({
        "Origen_Id": 1,
        "Fecha": dates,
        "target": np.arange(1, len(dates) + 1),
    })
    data = data.loc[data["Fecha"] != "2025-01-10"].reset_index(drop=True)
    data["day_of_week"] = data["Fecha"].dt.day_of_week
    return data


def test_lags_use_calendar_days():
    result = add_history_features(make_toy_data(), 1, [1, 7], [7])
    row = result.loc[result["Fecha"] == "2025-01-11"].iloc[0]
    assert np.isnan(row["lag_1"])
    assert row["lag_7"] == result.loc[result["Fecha"] == "2025-01-04", "target"].iloc[0]


def test_lag_shorter_than_horizon_fails():
    try:
        add_history_features(make_toy_data(), 3, [1, 7], [7])
    except ValueError:
        return
    raise AssertionError("Se esperaba un error con lag_1 y horizonte 3.")


def test_walk_forward_has_no_overlap():
    data = make_toy_data()
    for cutoff, train, validation in walk_forward_splits(data["Fecha"], "2025-01-01", 2, 1):
        assert data.loc[train, "Fecha"].max() < cutoff
        assert data.loc[validation, "Fecha"].min() >= cutoff
        assert not set(train) & set(validation)


def test_seasonal_naive_uses_last_week():
    data = add_history_features(make_toy_data(), 1, [7], [7])
    train, validation = data.iloc[:40], data.iloc[40:].copy()
    validation.loc[validation.index[0], "lag_7"] = np.nan
    predictions = predict_seasonal_naive(train, validation)
    expected_fallback = predict_station_weekday(train, validation).iloc[0]
    assert predictions.iloc[0] == expected_fallback
    has_lag = validation["lag_7"].notna()
    assert (predictions[has_lag] == validation.loc[has_lag, "lag_7"]).all()


def test_history_does_not_use_future_target():
    data = make_toy_data()
    original = add_history_features(data, 1, [1, 7], [7, 28])
    changed = data.copy()
    changed.loc[changed["Fecha"] >= "2025-01-20", "target"] = 99999
    updated = add_history_features(changed, 1, [1, 7], [7, 28])
    before = original["Fecha"] <= "2025-01-20"
    for column in ["lag_1", "lag_7", "roll_mean_7", "roll_mean_28"]:
        pd.testing.assert_series_equal(
            original.loc[before, column], updated.loc[before, column], check_names=False
        )


def test_duplicate_station_dates_fail():
    data = make_toy_data()
    repeated = pd.concat([data, data.iloc[[0]]], ignore_index=True)
    try:
        add_history_features(repeated, 1, [1, 7], [7])
    except ValueError:
        return
    raise AssertionError("Se esperaban rechazar fechas duplicadas por estación.")


def test_final_features_exclude_observed_weather():
    assert len(FEATURE_COLUMNS) == 18
    assert not set(WEATHER_COLUMNS).intersection(FEATURE_COLUMNS)
    assert "weather_zone" in FEATURE_COLUMNS


for test in [
    test_lags_use_calendar_days,
    test_lag_shorter_than_horizon_fails,
    test_walk_forward_has_no_overlap,
    test_seasonal_naive_uses_last_week,
    test_history_does_not_use_future_target,
    test_duplicate_station_dates_fail,
    test_final_features_exclude_observed_weather,
]:
    test()
    print(f"OK  {test.__name__}")

OK  test_lags_use_calendar_days
OK  test_lag_shorter_than_horizon_fails
OK  test_walk_forward_has_no_overlap
OK  test_seasonal_naive_uses_last_week
OK  test_history_does_not_use_future_target
OK  test_duplicate_station_dates_fail
OK  test_final_features_exclude_observed_weather


### 14.1 Pruebas de datos, archivos y predicción
Estas pruebas usan datos pequeños creados en memoria o archivos temporales. No vuelven a procesar los CSV originales ni entrenan el modelo completo.

Revisamos que el caché detecte cambios, que el inventario conserve las versiones de los CSV, que la integración no cambie la demanda y que el pipeline de Scikit-Learn acepte los datos esperados. También probamos casos con fechas duplicadas y estaciones nuevas.

Si una prueba falla, se detiene la celda para poder revisar el problema antes de continuar.

In [40]:
def test_cache_detects_changed_file():
    from tempfile import TemporaryDirectory

    with TemporaryDirectory() as folder:
        data_path = Path(folder) / "sample.parquet"
        data_path.write_bytes(b"first version")
        signature = {"period": "2020-2026", "schema": 1}
        save_cache_manifest(data_path, signature)
        assert cache_is_current(data_path, signature)
        assert not cache_is_current(data_path, {"period": "2021-2026", "schema": 1})
        data_path.write_bytes(b"changed version")
        assert not cache_is_current(data_path, signature)


def test_csv_versioning_keeps_history():
    from tempfile import TemporaryDirectory

    with TemporaryDirectory(dir=PROJECT_ROOT / "reports") as folder:
        root = Path(folder)
        source = root / "sources"
        versions = root / "versions"
        source.mkdir()
        csv_path = source / "sample.csv"
        csv_path.write_text("value\n1\n", encoding="utf-8")
        first = version_csv_sources(source, versions)
        assert len(first) == 1
        assert first.iloc[0]["version"] == 1
        assert version_csv_sources(source, versions).iloc[0]["known_versions"] == 1
        csv_path.write_text("value\n2\n", encoding="utf-8")
        second = version_csv_sources(source, versions)
        assert second.iloc[0]["version"] == 2
        assert second.iloc[0]["known_versions"] == 2
        assert csv_path.read_text(encoding="utf-8") == "value\n2\n"


def test_integration_rejects_changed_demand():
    daily = pd.DataFrame({
        "Origen_Id": [1, 1],
        "Fecha": pd.to_datetime(["2025-01-01", "2025-01-02"]),
        "target": [10, 20],
    })
    integrated = daily.copy()
    check_integration(daily, integrated)
    integrated.loc[0, "target"] = 11
    try:
        check_integration(daily, integrated)
    except ValueError:
        return
    raise AssertionError("La integración aceptó un cambio en los viajes.")


def test_split_by_date_keeps_test_separate():
    data = make_toy_data()
    development, test = split_by_date(data, pd.Timestamp("2025-01-01"))
    assert development["Fecha"].max() < pd.Timestamp("2025-01-01")
    assert test["Fecha"].min() >= pd.Timestamp("2025-01-01")
    assert len(development) + len(test) == len(data)


def test_pipeline_preprocessing_and_prediction():
    data = make_toy_data().copy()
    data["month"] = data["Fecha"].dt.month
    data["day_of_month"] = data["Fecha"].dt.day
    data["weather_zone"] = 1
    for column in POI_COLUMNS:
        data[column] = 0
    data = add_history_features(data, 1, LAGS, ROLLING_WINDOWS)
    columns = FEATURE_COLUMNS
    pipeline = build_model_pipeline(
        RandomForestRegressor(n_estimators=3, max_depth=2, random_state=SEED),
        columns,
    )
    pipeline.fit(data[columns], data[TARGET_COLUMN])
    predictions = pipeline.predict(data[columns].iloc[:3])
    assert len(predictions) == 3
    assert np.isfinite(predictions).all()


def test_next_day_rows_use_only_recent_stations():
    dates = pd.date_range("2025-01-01", periods=35)
    data = pd.DataFrame({
        "Origen_Id": [1] * len(dates) + [2],
        "Fecha": list(dates) + [pd.Timestamp("2024-01-01")],
    })
    for column in STATION_COLUMNS:
        data[column] = 0
    result = build_next_day_rows(data, pd.Timestamp("2025-02-05"), active_days=28)
    assert result["Origen_Id"].tolist() == [1]
    assert result["Fecha"].eq(pd.Timestamp("2025-02-05")).all()


additional_tests = [
    test_cache_detects_changed_file,
    test_csv_versioning_keeps_history,
    test_integration_rejects_changed_demand,
    test_split_by_date_keeps_test_separate,
    test_pipeline_preprocessing_and_prediction,
    test_next_day_rows_use_only_recent_stations,
]
for test in additional_tests:
    test()
    print(f"OK  {test.__name__}")

OK  test_cache_detects_changed_file
OK  test_csv_versioning_keeps_history
OK  test_integration_rejects_changed_demand
OK  test_split_by_date_keeps_test_separate
OK  test_pipeline_preprocessing_and_prediction
OK  test_next_day_rows_use_only_recent_stations


/home/josel/mlops/equipo-8-mibici-mlops/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


### 14.2 Evaluación y control del modelo
Antes de registrar una nueva versión como `champion`, revisamos estos puntos:

- **Datos:** los CSV tienen SHA-256, los Parquet tienen manifiesto y las fuentes externas están identificadas.
- **Pruebas:** pasan las comprobaciones de datos, variables históricas, separación temporal y predicción.
- **Resultados:** el modelo se compara con los baselines usando MAE y RMSE. Revisamos los errores por mes y por estaciones nuevas.
- **Reproducibilidad:** guardamos la configuración, las dependencias, la semilla y el commit de Git cuando está disponible.
- **Registro:** la versión del Model Registry apunta a la corrida correcta y el modelo puede cargarse para predecir.
- **Limitaciones:** los POI pueden no representar exactamente todos los años del estudio; la demanda futura también puede cambiar por cierres de estaciones o eventos que no están en los datos.

Las pruebas con datos pequeños detectan errores del código, pero no garantizan por sí solas la calidad de las predicciones. El resultado de 2026 se revisa por separado y no se utiliza para elegir hiperparámetros.

Por ahora no agregamos Docker, FastAPI, ONNX ni herramientas de gobernanza adicionales. Primero debemos comprobar que el pipeline y sus pruebas funcionen de principio a fin. Al trasladar las funciones a `.py`, estas pruebas pasarán a `tests/` para ejecutarlas con `pytest`.

## 15. Comprobación de equivalencia con el notebook 05
Comparamos los resultados de este notebook con los del 05. Como usamos los mismos datos, el mismo orden de filas, la misma semilla y los mismos parámetros, las diferencias deberían ser de redondeo (menores a 0.005). Si alguna fila sale con `False`, algo cambió en la refactorización y hay que revisarlo antes de pasar el código a los módulos.

In [41]:
test_by_segment = test_results.set_index("stations")
obtained = {
    **model_comparison["mae"].to_dict(),
    "Modelo final, 12 ventanas de 2025": final_walk_forward["mae"].mean(),
    "Modelo final, prueba 2026 (todas)": test_by_segment.loc["Todas", "model_mae"],
    "Ingenuo estacional, prueba 2026 (todas)": test_by_segment.loc["Todas", "naive_mae"],
    "Modelo final, prueba 2026 (nuevas)": (
        test_by_segment.loc["Nueva", "model_mae"] if "Nueva" in test_by_segment.index else np.nan
    ),
}
equivalence = pd.DataFrame({
    "notebook_05": pd.Series(REFERENCE_RESULTS),
    "notebook_06": pd.Series(obtained).reindex(list(REFERENCE_RESULTS)),
})
equivalence["difference"] = (equivalence["notebook_06"] - equivalence["notebook_05"]).abs()
equivalence["equivalent"] = equivalence["difference"] < 0.005
display(equivalence.round(3))
print(f"Resultados equivalentes: {int(equivalence['equivalent'].sum())} de {len(equivalence)}")

,notebook_05,notebook_06,difference,equivalent
Ingenuo estacional (lag_7),7.224,7.224,0.0,True
Promedio estación-día,7.703,7.703,0.0,True
Promedio global por día (versión anterior),22.986,22.986,0.0,True
HistGradientBoosting | calendario_estacion_historia,5.440,5.440,0.0,True
Random Forest | calendario_estacion_historia,5.456,5.456,0.0,True
"Modelo final, 12 ventanas de 2025",5.438,5.438,0.0,True
"Modelo final, prueba 2026 (todas)",5.896,5.896,0.0,True
"Ingenuo estacional, prueba 2026 (todas)",7.957,7.957,0.0,True
"Modelo final, prueba 2026 (nuevas)",3.511,3.511,0.0,True


Resultados equivalentes: 9 de 9


## 16. Pendientes para completar la Fase 2
1. Mover las funciones probadas a los módulos Python correspondientes. Este notebook todavía es un espacio de desarrollo; **no es la refactorización final** mientras las implementaciones sigan duplicadas aquí.
2. Sustituir gradualmente las definiciones por importaciones de los módulos y ejecutar desde un kernel limpio. Conservar las pruebas en `tests/` y ejecutarlas de manera independiente.
3. Comprobar que los resultados de la sección 15 coincidan con el notebook 05 usando los datos reales. No se ha validado esa equivalencia en este archivo.
4. Verificar la integridad de las fuentes externas y la fecha de captura de los POI; el manifiesto detecta cambios en archivos, pero no garantiza que el contenido externo represente el periodo histórico.
5. Comprobar en la instalación real de MLflow el registro, la versión y el alias del modelo, y revisar el experimento en la interfaz.
6. Actualizar `pyproject.toml`, `README.md`, evidencia de GitHub y presentación de la Fase 2.

**Modelo principal:** `calendario_estacion_historia` (18 variables), sin temperatura ni precipitación. `weather_zone` permanece como atributo de la estación. Las variables de clima observado solo se utilizan en experimentos opcionales.

Se incorporó un inventario SHA-256 de los CSV de `data/` y de los reportes de modelado; el manifiesto y las gráficas exploratorias se registran como artefactos de MLflow. Las fuentes grandes no se duplican.

**Control de calidad pendiente:** ejecutar todas las celdas con datos reales y comprobar que MLflow muestre los datasets en la corrida final, las gráficas en las corridas padre y las versiones del Model Registry. Las pruebas de este notebook no sustituyen una suite `pytest` ejecutada en CI.

### Interpretación de métricas en MLflow\nEn `documentation/EDA_METRICS_README.md` se explica cada indicador exploratorio, su unidad y qué representa. El archivo `eda/metric_series_guide.json` relaciona los pasos numéricos de MLflow con los meses, años y días reales. En `performance/README.md` se explica cómo interpretar los tiempos. MLflow muestra nombres de métricas, pero no incorpora estas descripciones directamente junto a cada valor; por eso se conservan ambos documentos en Artifacts. Los PNG originales de ambas fases se mantienen.\n

### Uso IA:

- **OpenAI. (2026). *ChatGPT*.** Utilizado como apoyo para organizar y refactorizar el código, revisar la redacción y las convenciones de nombres, desarrollar pruebas, revisar el pipeline de Scikit-Learn y el registro de experimentos en MLflow. También se utilizó para identificar errores y revisar la reproducibilidad de los resultados. https://chatgpt.com/